# Agricultural field dataset preparation

*Scaling Natural Flood Management on Agricultural Land in England*  
MSc dissertation code · Candidate number 1100992


## Notebook guide

This notebook creates the field-level agricultural input. Run the setup and reference geographies first, then FLAME attribution, CROME crop classes, ALC quality, Agricultural Census livestock context, physical availability and the final export. The main analysis notebook has the analysis and results for the three research questions.


This notebook writes `data/processed/agriculture.gpkg`.

## Setup and configuration


Set `NFM_DATA_ROOT` to a directory containing the `raw/` source files listed in the configuration cell. Use British National Grid (EPSG:27700) for area and distance calculations after checking each source CRS.


In [ ]:
# Install required packages
%pip install -q numpy pandas matplotlib shapely pyogrio geopandas rasterio

In [ ]:
# Import the packages and set up warnings and table display options.
import warnings, time, re, gc
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import defaultdict
import rasterio
from rasterio.mask import mask
from rasterio.features import shapes
from shapely.geometry import shape, MultiPolygon
from shapely import wkt
import tempfile
import json
import pyarrow.parquet as pq
from pyproj import CRS
import os
import uuid
import pyarrow as pa
import pyarrow.parquet as pq
import pyogrio
import shapely
import math
import sqlite3
try:
    from shapely import make_valid
    _HAS_MV = True
except Exception:
    _HAS_MV = False
warnings.filterwarnings("ignore", message=".*Geometry is in a geographic CRS.*")
warnings.filterwarnings("ignore", message=".*invalid value encountered.*")
pd.set_option("display.width", 120)
print("Packages ready.")

In [ ]:
# Set NFM_DATA_ROOT to a directory containing raw/ and processed/.
DATA_ROOT = Path("/Data")

# ---- Folders -----------------------------------------------------------------
RAW = DATA_ROOT / "raw"
OUT = DATA_ROOT / "processed";  OUT.mkdir(parents=True, exist_ok=True)
FIG = OUT / "figures";          FIG.mkdir(exist_ok=True)
TAB = OUT / "tables";           TAB.mkdir(exist_ok=True)

# ---- Analysis settings -------------------------------------------------------
ANALYSIS_CRS = "EPSG:27700"     # British National Grid
PURITY_MIN   = 0.50             # attribute counts as "confident" at >= this share
FORCE_REBUILD = False           # True = ignore cached checkpoints and rebuild every step from raw

# ---- All file paths ----------------------------------------------------------
PATHS = {
    # geographies
    "regions":      RAW / "Regions_England" / "RGN_DEC_2025_EN_BFC.shp",
    "local_auth":   RAW / "Local_Authority_England" / "LAD_MAY_2025_UK_BGC_V2.shp",
    "catchments":   RAW / "WFD_Surface_Water_Operational_Catchments_Cycle_4.shp/WFD_Surface_Water_Operational_Catchments_Cycle_4.shp",
    # field-level inputs
    "flame_farm":   RAW / "FLAME_field_polygons_with_farm.csv",
    "flame_owner": RAW / "FLAME_field_polygons_owner.csv",
    "crome":    RAW / "Crop_Map_of_England_CROME_2024.gpkg",
    "alc":          RAW / "Predictive_Agricultural_Land_Classification_Map_for_England" / "Predictive_ALC_England.shp",
    "agcensus_dir": RAW / "Agcensus_England",
    "rpa_points":   RAW / "RPA_Parcel_Points_England" / "RPA_Parcel_Points_England.shp",
    # exclusion layers
    "built_up":     RAW / "Built_Up_Areas" / "OS_Open_Built_Up_Extents.csv",
    "protected_dir":RAW / "Protected_Areas",
    "peat_tif":     RAW / "Peat" / "peaty_soil_depth_v1_download.tif",
    "road_dir":     RAW / "Transport" / "road",
    "rail":         RAW / "Transport" / "hotosm_gbr_railways_lines_shp.shp",
    "lakes":        RAW / "Water_Rivers" / "uklakes_v3_6_poly.gpkg",
    "watercourses": RAW / "Water_Rivers" / "WatercourseLink.shp",
    # Checkpoints
    "ck_flame": OUT / "flame_farm_owner_combined.parquet",
    "ck_crop":      OUT / "mp_flame_crop.parquet",
    "ck_alc":       OUT / "mp_flame_crop_alc.parquet",
    "ck_live":      OUT / "mp_flame_crop_alc_live_v2_age_grass.parquet",
    "exclusions":   OUT / "exclusions_england_streamed.parquet",
}
print("DATA_ROOT:", DATA_ROOT)
print("Outputs: ", OUT)

In [ ]:
# ---- Helper functions: spatial processing, summaries and outputs ----

def load_vector(path, layer=None, crs=ANALYSIS_CRS, bbox=None):
    """Read vector data and convert it to the analysis CRS."""
    try:
        g = gpd.read_file(path, layer=layer, bbox=bbox)
    except UnicodeDecodeError:
        # Retry with an alternative encoding for non-UTF-8 attribute text.
        g = gpd.read_file(path, layer=layer, bbox=bbox, encoding="latin1")
    return g.to_crs(crs)


def repair(gdf):
    """Repair geometries and retain only non-empty polygon parts."""
    gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()
    bad = ~gdf.geometry.is_valid

    if bad.any():
        gdf.loc[bad, "geometry"] = (
            gdf.loc[bad, "geometry"].apply(make_valid)
            if _HAS_MV else gdf.loc[bad, "geometry"].buffer(0)
        )

    # Repairs can produce lines or points, keep polygon parts for area calculations.
    gdf = gdf.explode(index_parts=False)
    gdf = gdf[gdf.geom_type.isin(["Polygon", "MultiPolygon"])]
    return gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].reset_index(drop=True)


def first_present(columns, candidates):
    """Return the first matching candidate column name, or None."""
    return next((c for c in candidates if c in columns), None)


def centroid_join(fields, polys, value_col):
    """Assign a polygon attribute using a point inside each field."""
    # Uses a representative point, not a centroid, crossing fields get one label.
    pts = fields[["field_id"]].copy()
    pts["geometry"] = fields.geometry.representative_point()
    pts = gpd.GeoDataFrame(pts, geometry="geometry", crs=fields.crs)
    j = (
        gpd.sjoin(
            pts, polys[[value_col, "geometry"]],
            how="left", predicate="within"
        )
        .groupby("field_id")[value_col].first()
    )
    return fields.drop(columns=[value_col], errors="ignore").merge(
        j, on="field_id", how="left"
    )


def dominant_by_area(fields, polygons, value_col, weight_col=None):
    """Assign each field the class with the greatest total overlap score."""
    keep = [c for c in [value_col, weight_col] if c] + ["geometry"]
    ov = gpd.overlay(
        fields[["field_id", "geometry"]], polygons[keep],
        how="intersection", keep_geom_type=False
    )
    if ov.empty:
        return pd.DataFrame(
            columns=["field_id", value_col, f"{value_col}_purity"]
        )

    ov["_a"] = ov.geometry.area
    tot = ov.groupby("field_id")["_a"].sum()

    # Sum overlap by class; optionally rank by area × weight.
    if weight_col:
        ov["_wa"] = ov["_a"] * ov[weight_col]
        agg = (
            ov.groupby(["field_id", value_col])
            .agg(_a=("_a", "sum"), _wa=("_wa", "sum"))
            .reset_index()
        )
        agg["_rank"] = agg["_wa"]
    else:
        agg = ov.groupby(["field_id", value_col])["_a"].sum().reset_index()
        agg["_rank"] = agg["_a"]

    win = agg.sort_values("_rank").drop_duplicates("field_id", keep="last")

    # Purity is the winning class's share of intersected area, not total field area.
    out = pd.DataFrame({
        "field_id": win["field_id"].values,
        value_col: win[value_col].values,
        f"{value_col}_purity": (
            win["_a"].values / win["field_id"].map(tot).values
        )
    })

    if weight_col:
        # Area-weighted mean weight for the winning class.
        out[f"{value_col}_conf"] = win["_wa"].values / win["_a"].values
    return out

def quick_look(gdf, name="layer"):
    """Print the layer name, number of features and CRS EPSG code."""
    crs = gdf.crs.to_epsg() if gdf.crs else None
    print(f"{name:<26} rows={len(gdf):>9,}  crs={crs}")


def save_table(df, name):
    """Save a CSV in TAB, print its filename and return the original table."""
    p = TAB / f"{name}.csv"
    df.to_csv(p, index=False)
    print("saved table:", p.name)
    return df


## Reference geographies


In [ ]:
# Load boundary layers, repair polygons and combine features by area name.

def load_boundaries(path, source_col, name_col):
    """Load and clean boundaries, returning one feature per named area."""
    gdf = (
        load_vector(path)
        .rename(columns={source_col: name_col})
        [[name_col, "geometry"]]
    )
    return (
        repair(gdf)
        .dissolve(by=name_col, as_index=False)
        [[name_col, "geometry"]]
    )


regions = load_boundaries(PATHS["regions"], "RGN25NM", "region")
local_auth = load_boundaries(PATHS["local_auth"], "LAD25NM", "local_authority")
catchments = load_boundaries(PATHS["catchments"], "operationa", "catchment")

In [ ]:
# Create the England boundary and select English local authorities.

england = regions[["geometry"]].dissolve().reset_index(drop=True)
england_geom = england.geometry.iloc[0]

# Select authorities using a point inside each feature, keeping boundaries intact.
local_auth_england = local_auth[
    local_auth.geometry.representative_point().within(england_geom)
].copy()

REGION_LIST = sorted(regions["region"].dropna().unique().tolist())

print("Regions to process:", REGION_LIST)
print("Local authorities in source:", len(local_auth))

quick_look(regions, "English regions")
quick_look(local_auth_england, "English local authorities")
quick_look(catchments, "Operational catchments")

## Field characteristics and availability


### FLAME fields and attribution


In [ ]:
# FLAME preprocessing —
RUN_FLAME_BUILD = False   # True: build the checkpoint if it is missing.
REBUILD_FLAME = False     # Also set True to replace an existing checkpoint.
BATCH_SIZE = 1_000
RAW_FLAME_CRS = "EPSG:27700"

combined_path = PATHS["ck_flame"]

# Keep the original attribute names for compatibility with later analysis.
schema = pa.schema(
    [
        ("ID", pa.string()),
        ("OWNER_ID", pa.string()),
        ("crop_old", pa.string()),
        ("area_flame", pa.float64()),
        ("flame_source", pa.string()),
        ("source_row_id", pa.int64()),
        ("field_id", pa.string()),
        ("entity_id", pa.string()),
        ("size_ha", pa.float64()),
        ("geometry", pa.binary()),
    ],
    metadata={
        b"geo": json.dumps({
            "version": "1.0.0",
            "primary_column": "geometry",
            "columns": {
                "geometry": {
                    "encoding": "WKB",
                    "geometry_types": [],
                    "crs": CRS.from_user_input(ANALYSIS_CRS).to_json_dict(),
                }
            },
        }).encode()
    },
)


def write_flame_source(writer, csv_path, source):
    """Read one CSV in batches and append standardised records to the output."""
    id_col = "ID" if source == "farm" else "OWNER_ID"
    columns = [id_col, "crop_maxA", "Area(hec)", "geometry"]
    count = 0

    with pd.read_csv(
        csv_path, usecols=columns, dtype="string", chunksize=BATCH_SIZE
    ) as batches:
        for batch_number, frame in enumerate(batches, start=1):
            frame = frame.reset_index(drop=True)

            # Parse raw geometry, transform coordinates and repair invalid shapes.
            geometry = (
                gpd.GeoSeries.from_wkt(frame["geometry"], crs=RAW_FLAME_CRS)
                .to_crs(ANALYSIS_CRS)
                .make_valid()
            )

            identifiers = frame[id_col]
            row_ids = pd.Series(range(count, count + len(frame)), dtype="int64")
            missing = pd.Series(pd.NA, index=frame.index, dtype="string")

            # Prefix identifiers to distinguish farm and owner source records.
            # Row-based field IDs remain stable only if CSV ordering is unchanged.
            data = pd.DataFrame({
                "ID": identifiers if source == "farm" else missing,
                "OWNER_ID": identifiers if source == "owner" else missing,
                "crop_old": frame["crop_maxA"],
                "area_flame": pd.to_numeric(
                    frame["Area(hec)"], errors="coerce"
                ).to_numpy(dtype=float, na_value=float("nan")),
                "flame_source": source,
                "source_row_id": row_ids,
                "field_id": "FLAME_" + source.upper() + "_" + row_ids.astype("string"),
                "entity_id": source.upper() + "_" + identifiers,
                "size_ha": geometry.area.to_numpy() / 10_000,
                "geometry": geometry.to_wkb().to_numpy(),
            })

            writer.write_table(
                pa.Table.from_pandas(data, schema=schema, preserve_index=False)
            )
            count += len(frame)

            if batch_number % 50 == 0:
                print(f"{source.upper()}: processed {count:,} records", flush=True)

    return count

# Stream both sources directly into one checkpoint, no source parquets are created.
if not RUN_FLAME_BUILD:
    print("Preprocessing skipped. Load the saved FLAME checkpoint below.")

elif combined_path.exists() and not REBUILD_FLAME:
    print("Using existing checkpoint:", combined_path)

else:
    sources = [
        ("farm", PATHS["flame_farm"]),
        ("owner", PATHS["flame_owner"]),
    ]
    for source, path in sources:
        if not path.exists():
            raise FileNotFoundError(f"Missing {source} CSV: {path}")

    combined_path.parent.mkdir(parents=True, exist_ok=True)

    # One temporary file protects the completed output if processing fails.
    # It is renamed on success and removed on failure.
    with tempfile.NamedTemporaryFile(
        dir=combined_path.parent, prefix="flame_build_", suffix=".tmp", delete=False
    ) as handle:
        temporary = Path(handle.name)

    try:
        total = 0
        with pq.ParquetWriter(temporary, schema, compression="snappy") as writer:
            for source, path in sources:
                count = write_flame_source(writer, path, source)
                total += count
                print(f"{source.upper()} complete: {count:,} records")

        if total == 0:
            raise ValueError("No FLAME records were read, checkpoint not replaced.")

        os.replace(temporary, combined_path)
        print(f"Saved {total:,} records to {combined_path}")

    finally:
        temporary.unlink(missing_ok=True)

In [ ]:
# Load the previously processed FLAME baseline.
# Run the optional preprocessing cell above only if this checkpoint is unavailable.
if not PATHS["ck_flame"].exists():
    raise FileNotFoundError(
        "FLAME checkpoint missing. Run the optional preprocessing cell first."
    )

flame = gpd.read_parquet(PATHS["ck_flame"])
print(f"Loaded {len(flame):,} FLAME records.")
flame.head()

In [ ]:
# Count spatial records and attributed entities from each FLAME source.
summary = flame.groupby("flame_source").agg(
    spatial_records=("field_id", "size"),
    attributed_entities=("entity_id", "nunique"),
)

display(summary.style.format("{:,.0f}"))

In [ ]:
# Check the processed FLAME layer - areas and IDs were created during preprocessing.
if flame.crs is None or flame.crs.to_epsg() != 27700:
    raise ValueError("Expected EPSG:27700, check the preprocessing output.")

ids = flame["field_id"].astype("string")
if ids.isna().any() or ids.str.strip().eq("").any() or not ids.is_unique:
    raise ValueError("Field IDs are missing or duplicated, check preprocessing.")

polygon = (
    flame.geometry.notna() & ~flame.geometry.is_empty
    & flame.geom_type.isin(["Polygon", "MultiPolygon"])
)
invalid = polygon & ~flame.geometry.is_valid
area = pd.to_numeric(flame["size_ha"], errors="coerce")

print(f"Missing, empty or non-polygon geometries: {(~polygon).sum():,}")
print(f"Invalid polygon geometries: {invalid.sum():,}")
print(f"Missing or non-positive areas: {(area.isna() | area.le(0)).sum():,}")
print(f"Records below 0.5 ha: {area.lt(0.5).sum():,}")
print(f"Records at least 0.5 ha: {area.ge(0.5).sum():,}")

display(flame[["field_id", "flame_source", "size_ha", "geometry"]].head())

#### Reporting geographies


In [ ]:
# Fast centroid joins (each field sits wholly inside one region/LA/catchment)
flame = centroid_join(flame, regions, "region")
flame = centroid_join(flame, local_auth, "local_authority")
flame = centroid_join(flame, catchments, "catchment")

# Rescue boundary fields whose centroid missed every region
missing = flame["region"].isna()
if missing.any():
    pts = flame.loc[missing, ["entity_id"]].copy()
    pts["geometry"] = flame.loc[missing].geometry.representative_point()
    pts = gpd.GeoDataFrame(pts, geometry="geometry", crs=flame.crs)
    near = gpd.sjoin_nearest(pts, regions[["region","geometry"]], how="left").groupby("entity_id")["region"].first()
    flame.loc[missing, "region"] = flame.loc[missing, "entity_id"].map(near)
print("Fields with no region after rescue:", int(flame["region"].isna().sum()))
print(flame["region"].value_counts(dropna=False))
flame.head()

### CROME crop classification


In [ ]:
# CROME 2024: detailed crop names and final land-use groups

CROME_LUCODE = {
    "AC01": "Spring Barley", "AC03": "Beet", "AC04": "Borage",
    "AC05": "Buckwheat", "AC06": "Canary Seed", "AC07": "Carrot",
    "AC09": "Chicory", "AC10": "Daffodil", "AC14": "Hemp",
    "AC15": "Lettuce", "AC16": "Spring Linseed", "AC17": "Maize",
    "AC18": "Millet", "AC19": "Spring Oats", "AC20": "Onions",
    "AC22": "Parsley", "AC23": "Parsnips", "AC24": "Spring Rye",
    "AC26": "Spinach", "AC27": "Strawberry",
    "AC30": "Spring Triticale", "AC32": "Spring Wheat",
    "AC34": "Spring Cabbage", "AC35": "Turnip",
    "AC36": "Spring Oilseed", "AC37": "Brown Mustard",
    "AC38": "Mustard", "AC41": "Radish", "AC44": "Potato",
    "AC45": "Tomato", "AC50": "Squash",
    "AC52": "Siam Pumpkin",
    "AC58": "Mixed Crop-Group 1",
    "AC59": "Mixed Crop-Group 2",
    "AC60": "Mixed Crop-Group 3",
    "AC61": "Mixed Crop-Group 4",
    "AC62": "Mixed Crop-Group 5",
    "AC63": "Winter Barley", "AC64": "Winter Linseed",
    "AC65": "Winter Oats", "AC66": "Winter Wheat",
    "AC67": "Winter Oilseed", "AC68": "Winter Rye",
    "AC69": "Winter Triticale", "AC70": "Winter Cabbage",
    "AC71": "Coriander", "AC72": "Corn Gromwell",
    "AC74": "Phacelia", "AC81": "Poppy",
    "AC88": "Sunflower", "AC90": "Gladioli",
    "AC92": "Sorghum", "AC94": "Sweet William",
    "AC100": "Italian Ryegrass",
    "AC00": "Unknown or Mixed Vegetation",
    "CA02": "Cover Crop",
    "LG01": "Chickpea", "LG02": "Fenugreek",
    "LG03": "Spring Field Beans", "LG04": "Green Beans",
    "LG06": "Lupins", "LG07": "Spring Peas",
    "LG08": "Soya", "LG09": "Cowpea",
    "LG11": "Lucerne", "LG13": "Sainfoin",
    "LG14": "Clover",
    "LG15": "Mixed Crops-Group 1 Leguminous",
    "LG16": "Mixed Crops-Group 2 Leguminous",
    "LG20": "Winter Field Beans",
    "LG21": "Winter Peas",
    "SR01": "Short Rotation Coppice",
    "FA01": "Fallow Land",
    "HE02": "Heathland and Bracken",
    "PG01": "Grass",
    "NA01": "Non-vegetated or Sparsely-Vegetated Land",
    "WA00": "Water",
    "TC01": "Perennial Crops and Isolated Trees",
    "NU01": "Nursery Crops",
    "WO12": "Trees and Scrubs, Short Woody Plants, Hedgerows",
}

# These are the final groups you supplied.
crop_groups = {
    "Cereal Crops": [
        "Spring Barley", "Beet", "Borage", "Buckwheat",
        "Canary Seed", "Carrot", "Celery", "Chicory",
        "Daffodil", "Dill", "Hemp", "Lettuce",
        "Spring Linseed", "Maize", "Millet", "Spring Oats",
        "Onions", "Parsley", "Parsnips", "Spring Rye",
        "Spinach", "Strawberry", "Spring Triticale",
        "Spring Wheat", "Spring Cabbage", "Turnip",
        "Spring Oilseed", "Brown Mustard", "Mustard",
        "Radish", "Potato", "Tomato", "Squash",
        "Siam Pumpkin", "Cheese Pumpkin",
        "Mixed Crop-Group 1", "Mixed Crop-Group 2",
        "Mixed Crop-Group 3", "Mixed Crop-Group 4",
        "Mixed Crop-Group 5",
        "Winter Barley", "Winter Linseed", "Winter Oats",
        "Winter Wheat", "Winter Oilseed", "Winter Rye",
        "Winter Triticale", "Winter Cabbage",
        "Coriander", "Corn Gromwell", "Phacelia",
        "German chamomile-type arable crops",
        "Corn Chamomile", "Poppy", "Hedge Bedstraw",
        "Sunflower", "Gladioli", "Sorghum", "Sweet William",
        "Italian Ryegrass", "Cover Crop",
    ],
    "Leguminous Crops": [
        "Chickpea", "Fenugreek", "Spring Field Beans",
        "Green Beans", "Lupins", "Spring Peas",
        "Soya", "Cowpea", "Lucerne", "Sainfoin", "Clover",
        "Mixed Crops-Group 1 Leguminous",
        "Mixed Crops-Group 2 Leguminous",
        "Mixed Crops-Group 3 Leguminous",
        "Winter Field Beans", "Winter Peas",
    ],
    "Grassland": [
        "Fallow Land", "Grass", "Temporary Grassland",
    ],
    "Non-Agricultural Land": [
        "Non-vegetated or Sparsely-Vegetated Land",
        "Non agriculture land",
        "Non-Agricultural Land",
        "Heather",
        "Heathland and Bracken",
    ],
    "Water": [
        "Water",
    ],
    "Trees": [
        "Perennial Crops and Isolated Trees",
        "Nursery Crops",
        "Trees and Scrubs, Short Woody Plants, Hedgerows",
        "Short Rotation Coppice",
    ],
}


def normalise_crop(series):
    return (
        series.astype("string")
        .str.normalize("NFKC")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .str.casefold()
        .replace("", pd.NA)
    )


crop_to_land_use = {
    normalise_crop(pd.Series([name])).iloc[0]: group
    for group, names in crop_groups.items()
    for name in names
}

label_fixes = {
    "wa00": "Water",
    "heat": "Heather",
}


def correct_crop_labels(series):
    values = series.astype("string")
    corrected = normalise_crop(values).map(label_fixes)
    return corrected.fillna(values)


def apply_final_crop_groups(frame):
    """Create the final groups directly on the FLAME dataset."""
    if "crop_type" not in frame.columns:
        raise KeyError("FLAME is missing crop_type.")

    if "crop_2020" not in frame.columns:
        if "crop_old" in frame.columns:
            frame["crop_2020"] = frame["crop_old"]
        else:
            frame["crop_2020"] = pd.NA

    for column in ["crop_old", "crop_2020", "crop_type"]:
        if column in frame.columns:
            frame[column] = correct_crop_labels(frame[column])

    frame["lu_flame"] = (
        normalise_crop(frame["crop_2020"])
        .map(crop_to_land_use)
    )

    frame["lu_crome"] = (
        normalise_crop(frame["crop_type"])
        .map(crop_to_land_use)
    )

    frame["land_use"] = frame["lu_crome"]

    if "crop_old" in frame.columns:
        frame["crop_old_group"] = (
            normalise_crop(frame["crop_old"])
            .map(crop_to_land_use)
        )

    return frame

In [ ]:
# CROME processing settings

CROME_LAYER = "Crop_Map_of_England_2024"

TILE_SIZE = 25_000
MAX_CROME_CELLS_PER_TILE = 1_000_000
MAX_CROME_SHARE_PER_TILE = 0.05

# Set True to inspect proposed tile sizes without intersections.
PREFLIGHT_ONLY = False

# Keep v3 so existing tile checkpoints can be reused: their
# detailed crop totals have not changed.
CROME_RUN_VERSION = "v3"

CROP_TILE_DIR = (
    OUT / f"crome_tiles_{CROME_RUN_VERSION}_{TILE_SIZE}"
)
CROP_TILE_DIR.mkdir(parents=True, exist_ok=True)

if not PATHS["crome"].exists():
    raise FileNotFoundError(
        f"CROME GeoPackage not found: {PATHS['crome']}"
    )

if flame.crs is None:
    raise ValueError("FLAME has no CRS.")

analysis_crs = CRS.from_user_input(ANALYSIS_CRS)

if CRS.from_user_input(flame.crs) != analysis_crs:
    print("Reprojecting FLAME to", ANALYSIS_CRS)
    flame = flame.to_crs(ANALYSIS_CRS)

if flame["field_id"].isna().any():
    raise ValueError("FLAME contains missing field_id values.")

if flame["field_id"].duplicated().any():
    raise ValueError("FLAME contains duplicate field_id values.")

crome_info = pyogrio.read_info(
    PATHS["crome"],
    layer=CROME_LAYER,
)

crome_crs = CRS.from_user_input(crome_info["crs"])
TOTAL_CROME_CELLS = int(crome_info["features"])

if crome_crs != analysis_crs:
    raise ValueError(
        f"CROME source CRS is {crome_crs}; "
        f"expected {analysis_crs}."
    )

print(f"FLAME records: {len(flame):,}")
print(f"CROME cells: {TOTAL_CROME_CELLS:,}")
print(f"Tile size: {TILE_SIZE / 1_000:g} km")
print("Tile checkpoint directory:", CROP_TILE_DIR)


# Determine whether probability is stored as 0–1 or 0–100.
probability_sample = pyogrio.read_dataframe(
    PATHS["crome"],
    layer=CROME_LAYER,
    columns=["prob"],
    read_geometry=False,
    max_features=10_000,
)

probability_sample["prob"] = pd.to_numeric(
    probability_sample["prob"],
    errors="coerce",
)

sample_probability_max = probability_sample["prob"].max()

CROME_PROBABILITY_DIVISOR = (
    100.0
    if pd.notna(sample_probability_max)
    and sample_probability_max > 1
    else 1.0
)

print(
    "CROME probability divisor:",
    CROME_PROBABILITY_DIVISOR,
)


def gpkg_bbox_count(path, layer, bbox):
    """Count candidate cells through the GeoPackage spatial index."""
    x0, y0, x1, y1 = bbox

    with sqlite3.connect(
        f"file:{path}?mode=ro",
        uri=True,
    ) as connection:
        metadata = connection.execute(
            """
            SELECT table_name, column_name
            FROM gpkg_geometry_columns
            WHERE table_name = ?
            """,
            (layer,),
        ).fetchone()

        if metadata is None:
            raise RuntimeError(
                f"Layer {layer!r} was not found."
            )

        table_name, geometry_column = metadata
        rtree_name = f"rtree_{table_name}_{geometry_column}"

        rtree_exists = connection.execute(
            """
            SELECT COUNT(*)
            FROM sqlite_master
            WHERE type = 'table' AND name = ?
            """,
            (rtree_name,),
        ).fetchone()[0]

        if not rtree_exists:
            raise RuntimeError(
                f"GeoPackage spatial index {rtree_name!r} "
                "is missing."
            )

        quoted_rtree = (
            '"' + rtree_name.replace('"', '""') + '"'
        )

        count = connection.execute(
            f"""
            SELECT COUNT(*)
            FROM {quoted_rtree}
            WHERE minx <= ?
              AND maxx >= ?
              AND miny <= ?
              AND maxy >= ?
            """,
            (x1, x0, y1, y0),
        ).fetchone()[0]

    return int(count)


def save_tile_checkpoint(frame, destination):
    """Avoid treating an interrupted write as a completed tile."""
    temporary = destination.with_suffix(
        ".temporary.parquet"
    )
    frame.to_parquet(temporary, index=False)
    temporary.replace(destination)

In [ ]:
# Build tile grid
xmin, ymin, xmax, ymax = regions.total_bounds

xmin = math.floor(xmin / TILE_SIZE) * TILE_SIZE
ymin = math.floor(ymin / TILE_SIZE) * TILE_SIZE
xmax = math.ceil(xmax / TILE_SIZE) * TILE_SIZE
ymax = math.ceil(ymax / TILE_SIZE) * TILE_SIZE

CROME_TILES = [
    (x0, y0)
    for y0 in range(int(ymin), int(ymax), TILE_SIZE)
    for x0 in range(int(xmin), int(xmax), TILE_SIZE)
]

print(f"Constructed {len(CROME_TILES):,} tiles.")

BUILD_CROP_RESULTS = (
    FORCE_REBUILD
    or not PATHS["ck_crop"].exists()
)


In [ ]:
# Use existing completed checkpoint when available
if not BUILD_CROP_RESULTS:
    flame = gpd.read_parquet(PATHS["ck_crop"])

    required_columns = {
        "field_id",
        "crop_type",
        "crop_type_purity",
        "crop_prob",
        "geometry",
    }

    missing_columns = required_columns - set(flame.columns)

    if missing_columns:
        raise RuntimeError(
            "Existing crop checkpoint is incomplete. "
            "Missing columns: "
            + ", ".join(sorted(missing_columns))
        )

    flame = apply_final_crop_groups(flame)
    flame.to_parquet(PATHS["ck_crop"])

    print(
        "Updated groups in existing crop checkpoint:",
        PATHS["ck_crop"],
    )

In [ ]:
# Read, intersect, and checkpoint CROME tiles

preflight_records = []

if BUILD_CROP_RESULTS:

    for tile_number, (x0, y0) in enumerate(
        CROME_TILES,
        start=1,
    ):
        x1 = x0 + TILE_SIZE
        y1 = y0 + TILE_SIZE

        tile_path = (
            CROP_TILE_DIR / f"crop_{x0}_{y0}.parquet"
        )

        if (
            tile_path.exists()
            and not FORCE_REBUILD
            and not PREFLIGHT_ONLY
        ):
            print(
                f"[{tile_number}/{len(CROME_TILES)}] "
                f"{x0},{y0}: checkpoint exists"
            )
            continue

        tile_geometry = box(x0, y0, x1, y1)

        f_tile = flame.cx[x0:x1, y0:y1][
            ["field_id", "geometry"]
        ].copy()

        if f_tile.empty:
            continue

        invalid_fields = ~f_tile.geometry.is_valid

        if invalid_fields.any():
            f_tile.loc[invalid_fields, "geometry"] = (
                f_tile.loc[invalid_fields, "geometry"]
                .make_valid()
            )

        f_tile["geometry"] = (
            f_tile.geometry.intersection(tile_geometry)
        )

        f_tile = f_tile.loc[
            f_tile.geometry.notna()
            & ~f_tile.geometry.is_empty
            & (f_tile.geometry.area > 0)
        ].copy()

        if f_tile.empty:
            del f_tile
            gc.collect()
            continue

        bbox = (x0, y0, x1, y1)

        expected_count = gpkg_bbox_count(
            PATHS["crome"],
            CROME_LAYER,
            bbox,
        )

        expected_share = (
            expected_count / TOTAL_CROME_CELLS
        )

        preflight_records.append({
            "tile_number": tile_number,
            "x0": x0,
            "y0": y0,
            "field_pieces": len(f_tile),
            "expected_crome_cells": expected_count,
            "national_share": expected_share,
        })

        print(
            f"[{tile_number}/{len(CROME_TILES)}] "
            f"tile={x0},{y0} | "
            f"fields={len(f_tile):,} | "
            f"expected CROME={expected_count:,} "
            f"({expected_share:.3%})",
            flush=True,
        )

        if (
            expected_count > MAX_CROME_CELLS_PER_TILE
            or expected_share > MAX_CROME_SHARE_PER_TILE
        ):
            raise RuntimeError(
                "\nSAFETY STOP\n"
                f"Tile: {x0},{y0}\n"
                f"Expected CROME cells: {expected_count:,}\n"
                f"National share: {expected_share:.2%}\n"
                "Reduce TILE_SIZE and choose a new "
                "CROME_RUN_VERSION."
            )

        if PREFLIGHT_ONLY:
            del f_tile
            gc.collect()
            continue

        started = time.time()

        c_tile = gpd.read_file(
            PATHS["crome"],
            layer=CROME_LAYER,
            columns=["lucode", "prob"],
            bbox=bbox,
            engine="pyogrio",
            use_arrow=True,
        )

        if len(c_tile) > MAX_CROME_CELLS_PER_TILE:
            raise RuntimeError(
                f"SAFETY STOP: tile {x0},{y0} loaded "
                f"{len(c_tile):,} CROME cells."
            )

        if c_tile.empty:
            tile_result = pd.DataFrame({
                "field_id": pd.Series(
                    dtype=flame["field_id"].dtype
                ),
                "crop_type": pd.Series(dtype="string"),
                "_area": pd.Series(dtype="float64"),
                "_weighted_area": pd.Series(dtype="float64"),
            })

            save_tile_checkpoint(tile_result, tile_path)
            del f_tile, c_tile, tile_result
            gc.collect()
            continue

        c_tile["crop_type"] = (
            c_tile["lucode"]
            .astype("string")
            .str.strip()
            .str.upper()
            .map(CROME_LUCODE)
        )

        c_tile["crome_prob"] = (
            pd.to_numeric(
                c_tile["prob"],
                errors="coerce",
            )
            / CROME_PROBABILITY_DIVISOR
        )

        c_tile = c_tile.loc[
            c_tile["crop_type"].notna()
            & c_tile["crome_prob"].notna()
            & np.isfinite(c_tile["crome_prob"]),
            ["crop_type", "crome_prob", "geometry"],
        ].copy()

        if c_tile.empty:
            tile_result = pd.DataFrame({
                "field_id": pd.Series(
                    dtype=flame["field_id"].dtype
                ),
                "crop_type": pd.Series(dtype="string"),
                "_area": pd.Series(dtype="float64"),
                "_weighted_area": pd.Series(dtype="float64"),
            })
        else:
            intersections = gpd.overlay(
                f_tile,
                c_tile,
                how="intersection",
                keep_geom_type=False,
                make_valid=True,
            )

            if intersections.empty:
                tile_result = pd.DataFrame({
                    "field_id": pd.Series(
                        dtype=flame["field_id"].dtype
                    ),
                    "crop_type": pd.Series(dtype="string"),
                    "_area": pd.Series(dtype="float64"),
                    "_weighted_area": pd.Series(dtype="float64"),
                })
            else:
                intersections["_area"] = (
                    intersections.geometry.area
                )

                intersections = intersections.loc[
                    np.isfinite(intersections["_area"])
                    & intersections["_area"].gt(0)
                ].copy()

                intersections["_weighted_area"] = (
                    intersections["_area"]
                    * intersections["crome_prob"]
                )

                tile_result = (
                    intersections
                    .groupby(
                        ["field_id", "crop_type"],
                        as_index=False,
                        sort=False,
                    )
                    .agg(
                        _area=("_area", "sum"),
                        _weighted_area=(
                            "_weighted_area", "sum"
                        ),
                    )
                )

                del intersections

        save_tile_checkpoint(tile_result, tile_path)

        print(
            f"    saved {len(tile_result):,} "
            f"field/crop totals in "
            f"{time.time() - started:.0f}s",
            flush=True,
        )

        del f_tile, c_tile, tile_result
        gc.collect()


preflight_table = pd.DataFrame(preflight_records)

if not preflight_table.empty:
    preflight_table.to_csv(
        OUT
        / (
            f"crome_preflight_"
            f"{CROME_RUN_VERSION}_{TILE_SIZE}.csv"
        ),
        index=False,
    )

    display(
        preflight_table
        .sort_values(
            "expected_crome_cells",
            ascending=False,
        )
        .head(20)
    )

if PREFLIGHT_ONLY and BUILD_CROP_RESULTS:
    print(
        "PREFLIGHT ONLY completed. "
        "No CROME intersections were performed."
    )


In [ ]:
# Combine tile totals and save the completed crop checkpoint

if BUILD_CROP_RESULTS and not PREFLIGHT_ONLY:

    tile_files = sorted(
        CROP_TILE_DIR.glob("crop_*.parquet")
    )

    if not tile_files:
        raise RuntimeError(
            "No CROME tile checkpoints were produced."
        )

    print(
        f"Combining {len(tile_files):,} "
        "tile checkpoints..."
    )

    crop_totals = pd.concat(
        [pd.read_parquet(path) for path in tile_files],
        ignore_index=True,
    )

    if crop_totals.empty:
        raise RuntimeError(
            "All CROME tile results are empty."
        )

    crop_totals = (
        crop_totals
        .groupby(
            ["field_id", "crop_type"],
            as_index=False,
            sort=False,
        )
        .agg(
            _area=("_area", "sum"),
            _weighted_area=("_weighted_area", "sum"),
        )
    )

    crop_totals["_total_intersected_area"] = (
        crop_totals
        .groupby("field_id")["_area"]
        .transform("sum")
    )

    crop_totals["_purity"] = (
        crop_totals["_area"]
        / crop_totals["_total_intersected_area"]
    )

    winners = (
        crop_totals
        .sort_values(
            [
                "field_id",
                "_weighted_area",
                "_area",
                "crop_type",
            ]
        )
        .drop_duplicates("field_id", keep="last")
        .copy()
    )

    crop_result = pd.DataFrame({
        "field_id": winners["field_id"],
        "crop_type": winners["crop_type"],
        "crop_type_purity": winners["_purity"],
        "crop_prob": (
            winners["_weighted_area"]
            / winners["_area"]
        ),
    })

    if crop_result["field_id"].duplicated().any():
        raise RuntimeError(
            "Combined CROME result contains "
            "duplicate field IDs."
        )

    flame = (
        flame
        .drop(
            columns=[
                "crop_type",
                "crop_type_purity",
                "crop_prob",
                "crop_source",
                "lu_flame",
                "lu_crome",
                "land_use",
                "crop_old_group",
                "_tile_x",
                "_tile_y",
            ],
            errors="ignore",
        )
        .merge(
            crop_result,
            on="field_id",
            how="left",
        )
    )

    flame["crop_source"] = "CROME 2024"
    flame = apply_final_crop_groups(flame)

    flame.to_parquet(PATHS["ck_crop"])

    print(
        "Saved final crop checkpoint:",
        PATHS["ck_crop"],
    )

    del crop_totals, winners, crop_result
    gc.collect()

In [ ]:
# Final checks

if not (BUILD_CROP_RESULTS and PREFLIGHT_ONLY):

    checks = {
        "Unique field IDs":
            flame["field_id"].is_unique,

        "No missing field IDs":
            flame["field_id"].notna().all(),

        "Purity within 0–1":
            flame["crop_type_purity"]
            .dropna()
            .between(0, 1)
            .all(),

        "Probability within 0–1":
            flame["crop_prob"]
            .dropna()
            .between(0, 1)
            .all(),

        "land_use equals lu_crome":
            flame["land_use"]
            .fillna("__missing__")
            .eq(
                flame["lu_crome"]
                .fillna("__missing__")
            )
            .all(),
    }

    for name, passed in checks.items():
        print(
            f"{'PASS' if passed else 'FAIL'}: {name}"
        )

    matched = flame["crop_type"].notna()

    print("\nResult summary")
    print("------------------------------")
    print(f"Fields:             {len(flame):,}")
    print(f"CROME matched:      {matched.sum():,}")
    print(f"CROME match rate:   {matched.mean():.2%}")
    print(f"Missing crop:       {(~matched).sum():,}")
    print(
        "Median purity:     ",
        round(flame["crop_type_purity"].median(), 3),
    )
    print(
        "Median probability:",
        round(flame["crop_prob"].median(), 3),
    )

    print("\nFinal land-use groups:")
    print(
        flame["land_use"]
        .value_counts(dropna=False)
        .to_string()
    )

    unmapped = (
        flame["crop_type"].notna()
        & flame["land_use"].isna()
    )

    print("\nCROME crop names without a group:")
    print(
        flame.loc[unmapped, "crop_type"]
        .value_counts()
        .to_string()
    )

In [ ]:
# Crop labels to analytical land-use groups
crop_to_land_use = {
    # --------------------------------------------------------
    # Arable
    # --------------------------------------------------------
    "Spring Barley": "Arable",
    "Beet": "Arable",
    "Borage": "Arable",
    "Buckwheat": "Arable",
    "Canary Seed": "Arable",
    "Chicory": "Arable",
    "Hemp": "Arable",
    "Spring Linseed": "Arable",
    "Maize": "Arable",
    "Millet": "Arable",
    "Spring Oats": "Arable",
    "Spring Rye": "Arable",
    "Spring Triticale": "Arable",
    "Spring Wheat": "Arable",
    "Spring Oilseed": "Arable",
    "Brown Mustard": "Arable",
    "Mustard": "Arable",
    "Winter Barley": "Arable",
    "Winter Linseed": "Arable",
    "Winter Oats": "Arable",
    "Winter Wheat": "Arable",
    "Winter Oilseed": "Arable",
    "Winter Rye": "Arable",
    "Winter Triticale": "Arable",
    "Coriander": "Arable",
    "Corn Gromwell": "Arable",
    "Phacelia": "Arable",
    "Poppy": "Arable",
    "Sunflower": "Arable",
    "Sorghum": "Arable",
    "Chickpea": "Arable",
    "Fenugreek": "Arable",
    "Spring Field Beans": "Arable",
    "Lupins": "Arable",
    "Spring Peas": "Arable",
    "Soya": "Arable",
    "Cowpea": "Arable",
    "Winter Field Beans": "Arable",
    "Winter Peas": "Arable",
    "Short Rotation Coppice": "Arable",

    # Mixed agricultural crop classes
    "Mixed Crop-Group 1": "Arable",
    "Mixed Crop-Group 2": "Arable",
    "Mixed Crop-Group 3": "Arable",
    "Mixed Crop-Group 4": "Arable",
    "Mixed Crop-Group 5": "Arable",
    "Mixed Crops-Group 1 Leguminous": "Arable",
    "Mixed Crops-Group 2 Leguminous": "Arable",

    # --------------------------------------------------------
    # Potatoes and horticulture
    # --------------------------------------------------------
    "Carrot": "Potatoes and horticulture",
    "Daffodil": "Potatoes and horticulture",
    "Lettuce": "Potatoes and horticulture",
    "Onions": "Potatoes and horticulture",
    "Parsley": "Potatoes and horticulture",
    "Parsnips": "Potatoes and horticulture",
    "Spinach": "Potatoes and horticulture",
    "Strawberry": "Potatoes and horticulture",
    "Spring Cabbage": "Potatoes and horticulture",
    "Turnip": "Potatoes and horticulture",
    "Radish": "Potatoes and horticulture",
    "Potato": "Potatoes and horticulture",
    "Tomato": "Potatoes and horticulture",
    "Squash": "Potatoes and horticulture",
    "Siam Pumpkin": "Potatoes and horticulture",
    "Winter Cabbage": "Potatoes and horticulture",
    "Gladioli": "Potatoes and horticulture",
    "Sweet William": "Potatoes and horticulture",
    "Green Beans": "Potatoes and horticulture",
    "Perennial Crops and Isolated Trees":
        "Potatoes and horticulture",
    "Nursery Crops": "Potatoes and horticulture",

    # --------------------------------------------------------
    # Temporary grass and forage
    # --------------------------------------------------------
    "Italian Ryegrass": "Temporary grass and forage",
    "Cover Crop": "Temporary grass and forage",
    "Lucerne": "Temporary grass and forage",
    "Sainfoin": "Temporary grass and forage",
    "Clover": "Temporary grass and forage",

    # --------------------------------------------------------
    # Permanent grass
    # --------------------------------------------------------
    "Grass": "Permanent grass",

    # --------------------------------------------------------
    # Fallow
    # --------------------------------------------------------
    "Fallow Land": "Fallow or uncropped land",

    # --------------------------------------------------------
    # Separate reporting categories
    # --------------------------------------------------------
    "Unknown or Mixed Vegetation": "Unrecognised",

    "Heathland and Bracken": "Non-agricultural",

    "Non-vegetated or Sparsely-Vegetated Land":
        "Non-agricultural",

    "Water": "Non-agricultural",

    "Trees and Scrubs, Short Woody Plants, Hedgerows":
        "Non-agricultural",
}

In [ ]:
# Cache the whole CROME layer once
if (OUT / "crome_all_2024_gpkg.parquet").exists() and not FORCE_REBUILD:
    crome_all = gpd.read_parquet(OUT / "crome_all_2024_gpkg.parquet")
    print(f"Loaded CROME from cache: {len(crome_all):,} cells")
else:
    t0 = time.time()
    crome_all = gpd.read_file(PATHS["crome_gdb"])
    crome_all["crop_type"] = (crome_all["lucode"].astype("string").str.strip().str.upper().map(CROME_LUCODE))
    crome_all = crome_all.rename(columns={"prob":"crome_prob"})[["crop_type","crome_prob","geometry"]].to_crs(ANALYSIS_CRS)
    crome_all.to_parquet(OUT / "crome_all.parquet")
    print(f"Cached CROME: {len(crome_all):,} cells in {time.time()-t0:.0f}s")

In [ ]:
# CROME 2024 upload
CROME_LAYER = "Crop_Map_of_England_2024"

# Start with 25 km to ensure RAM is not exceeded.
TILE_SIZE = 25_000
MAX_CROME_CELLS_PER_TILE = 1_000_000
MAX_CROME_SHARE_PER_TILE = 0.05

# True: only count proposed tile sizes, do no intersections.
# False: perform the complete processing.
PREFLIGHT_ONLY = False

# Change the version if processing logic changes.
CROME_RUN_VERSION = "v3"

CROP_TILE_DIR = (
    OUT
    / f"crome_tiles_{CROME_RUN_VERSION}_{TILE_SIZE}"
)

CROP_TILE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

if not PATHS["crome"].exists():
    raise FileNotFoundError(
        f"CROME GeoPackage not found: {PATHS['crome']}"
    )

if flame.crs is None:
    raise ValueError("FLAME has no CRS.")

analysis_crs = CRS.from_user_input(ANALYSIS_CRS)

if CRS.from_user_input(flame.crs) != analysis_crs:
    print("Reprojecting FLAME to", ANALYSIS_CRS)
    flame = flame.to_crs(ANALYSIS_CRS)

if flame["field_id"].isna().any():
    raise ValueError("FLAME contains missing field_id values.")

if flame["field_id"].duplicated().any():
    raise ValueError("FLAME contains duplicate field_id values.")

crome_info = pyogrio.read_info(
    PATHS["crome"],
    layer=CROME_LAYER,
)

crome_crs = CRS.from_user_input(crome_info["crs"])
TOTAL_CROME_CELLS = int(crome_info["features"])

if crome_crs != analysis_crs:
    raise ValueError(
        f"CROME source CRS is {crome_crs}; "
        f"expected {analysis_crs}."
    )

print(f"FLAME records: {len(flame):,}")
print(f"CROME cells: {TOTAL_CROME_CELLS:,}")
print(f"Tile size: {TILE_SIZE / 1_000:g} km")
print("CROME CRS:", crome_crs)
print("Tile checkpoint directory:", CROP_TILE_DIR)

In [ ]:
# CROME helpers

# Determine whether CROME probability is stored as 0–1 or 0–100.
probability_sample = pyogrio.read_dataframe(
    PATHS["crome"],
    layer=CROME_LAYER,
    columns=["prob"],
    read_geometry=False,
    max_features=10_000,
)

probability_sample["prob"] = pd.to_numeric(
    probability_sample["prob"],
    errors="coerce",
)

sample_probability_max = probability_sample["prob"].max()

CROME_PROBABILITY_DIVISOR = (
    100.0
    if pd.notna(sample_probability_max)
    and sample_probability_max > 1
    else 1.0
)

print(
    "CROME probability divisor:",
    CROME_PROBABILITY_DIVISOR,
)


def gpkg_bbox_count(path, layer, bbox):
    """
    Count candidate geometries using the GeoPackage R-tree.

    This does not load the geometries into Python.
    """
    x0, y0, x1, y1 = bbox

    with sqlite3.connect(
        f"file:{path}?mode=ro",
        uri=True,
    ) as connection:

        geometry_metadata = connection.execute(
            """
            SELECT table_name, column_name
            FROM gpkg_geometry_columns
            WHERE table_name = ?
            """,
            (layer,),
        ).fetchone()

        if geometry_metadata is None:
            raise RuntimeError(
                f"Layer {layer!r} was not found."
            )

        table_name, geometry_column = geometry_metadata
        rtree_name = (
            f"rtree_{table_name}_{geometry_column}"
        )

        rtree_exists = connection.execute(
            """
            SELECT COUNT(*)
            FROM sqlite_master
            WHERE type = 'table'
              AND name = ?
            """,
            (rtree_name,),
        ).fetchone()[0]

        if not rtree_exists:
            raise RuntimeError(
                f"GeoPackage spatial index {rtree_name!r} "
                "is missing. Processing has been stopped."
            )

        quoted_rtree = (
            '"'
            + rtree_name.replace('"', '""')
            + '"'
        )

        candidate_count = connection.execute(
            f"""
            SELECT COUNT(*)
            FROM {quoted_rtree}
            WHERE minx <= ?
              AND maxx >= ?
              AND miny <= ?
              AND maxy >= ?
            """,
            (x1, x0, y1, y0),
        ).fetchone()[0]

    return int(candidate_count)


def save_tile_checkpoint(frame, destination):
    """Write safely so an interrupted write is not treated as complete."""
    temporary = destination.with_suffix(
        ".temporary.parquet"
    )

    frame.to_parquet(
        temporary,
        index=False,
    )

    temporary.replace(destination)

In [ ]:
# Construct the England processing grid
# Use the official regions extent, not FLAME total_bounds.
xmin, ymin, xmax, ymax = regions.total_bounds

xmin = math.floor(xmin / TILE_SIZE) * TILE_SIZE
ymin = math.floor(ymin / TILE_SIZE) * TILE_SIZE
xmax = math.ceil(xmax / TILE_SIZE) * TILE_SIZE
ymax = math.ceil(ymax / TILE_SIZE) * TILE_SIZE

CROME_TILES = [
    (x0, y0)
    for y0 in range(
        int(ymin),
        int(ymax),
        TILE_SIZE,
    )
    for x0 in range(
        int(xmin),
        int(xmax),
        TILE_SIZE,
    )
]

print(
    f"Constructed {len(CROME_TILES):,} "
    f"{TILE_SIZE / 1_000:g} km tiles."
)

BUILD_CROP_RESULTS = (
    FORCE_REBUILD
    or not PATHS["ck_crop"].exists()
)

if not BUILD_CROP_RESULTS:
    flame = gpd.read_parquet(
        PATHS["ck_crop"]
    )

    required_checkpoint_columns = {
        "field_id",
        "crop_type",
        "crop_type_purity",
        "crop_prob",
        "geometry",
    }

    missing_checkpoint_columns = (
        required_checkpoint_columns
        - set(flame.columns)
    )

    if missing_checkpoint_columns:
        raise RuntimeError(
            "Existing crop checkpoint is incomplete. "
            "Missing columns: "
            + ", ".join(
                sorted(missing_checkpoint_columns)
            )
        )

    print(
        "Loaded completed crop checkpoint:",
        PATHS["ck_crop"],
    )
else:
    print("Crop result must be built.")

In [ ]:
# Load, intersect and checkpoint each CROME tile
preflight_records = []

if BUILD_CROP_RESULTS:

    for tile_number, (x0, y0) in enumerate(
        CROME_TILES,
        start=1,
    ):
        x1 = x0 + TILE_SIZE
        y1 = y0 + TILE_SIZE

        tile_path = (
            CROP_TILE_DIR
            / f"crop_{x0}_{y0}.parquet"
        )

        if (
            tile_path.exists()
            and not FORCE_REBUILD
            and not PREFLIGHT_ONLY
        ):
            print(
                f"[{tile_number}/{len(CROME_TILES)}] "
                f"{x0},{y0}: checkpoint exists"
            )
            continue

        tile_geometry = box(
            x0,
            y0,
            x1,
            y1,
        )

        f_tile = flame.cx[
            x0:x1,
            y0:y1,
        ][
            ["field_id", "geometry"]
        ].copy()

        if f_tile.empty:
            continue

        # Repair only geometries needed in this tile.
        invalid_fields = ~f_tile.geometry.is_valid

        if invalid_fields.any():
            f_tile.loc[
                invalid_fields,
                "geometry",
            ] = (
                f_tile.loc[
                    invalid_fields,
                    "geometry",
                ]
                .make_valid()
            )

        f_tile["geometry"] = (
            f_tile.geometry.intersection(
                tile_geometry
            )
        )

        f_tile = f_tile.loc[
            f_tile.geometry.notna()
            & ~f_tile.geometry.is_empty
            & (f_tile.geometry.area > 0)
        ].copy()

        if f_tile.empty:
            del f_tile
            gc.collect()
            continue

        bbox = (x0, y0, x1, y1)

        expected_count = gpkg_bbox_count(
            PATHS["crome"],
            CROME_LAYER,
            bbox,
        )

        expected_share = (
            expected_count
            / TOTAL_CROME_CELLS
        )

        preflight_records.append({
            "tile_number": tile_number,
            "x0": x0,
            "y0": y0,
            "field_pieces": len(f_tile),
            "expected_crome_cells": expected_count,
            "national_share": expected_share,
        })

        print(
            f"[{tile_number}/{len(CROME_TILES)}] "
            f"tile={x0},{y0} | "
            f"fields={len(f_tile):,} | "
            f"expected CROME={expected_count:,} "
            f"({expected_share:.3%})",
            flush=True,
        )

        # Stop before attempting an oversized read.
        if (
            expected_count
            > MAX_CROME_CELLS_PER_TILE
            or expected_share
            > MAX_CROME_SHARE_PER_TILE
        ):
            raise RuntimeError(
                "\nSAFETY STOP\n"
                f"Tile: {x0},{y0}\n"
                f"Expected CROME cells: "
                f"{expected_count:,}\n"
                f"National share: "
                f"{expected_share:.2%}\n"
                f"Current TILE_SIZE: "
                f"{TILE_SIZE:,} metres\n"
                "Reduce TILE_SIZE and use a new "
                "CROME_RUN_VERSION."
            )

        if PREFLIGHT_ONLY:
            del f_tile
            gc.collect()
            continue

        started = time.time()

        # Only now load this tile from the GeoPackage.
        c_tile = gpd.read_file(
            PATHS["crome"],
            layer=CROME_LAYER,
            columns=["lucode", "prob"],
            bbox=bbox,
            engine="pyogrio",
            use_arrow=True,
        )

        # Verify actual read size before overlay.
        if len(c_tile) > MAX_CROME_CELLS_PER_TILE:
            actual_count = len(c_tile)

            del f_tile, c_tile
            gc.collect()

            raise RuntimeError(
                f"SAFETY STOP: tile {x0},{y0} "
                f"actually loaded {actual_count:,} "
                "CROME cells."
            )

        if c_tile.empty:
            empty_result = pd.DataFrame({
                "field_id": pd.Series(
                    dtype=flame["field_id"].dtype
                ),
                "crop_type": pd.Series(
                    dtype="string"
                ),
                "_area": pd.Series(dtype="float64"),
                "_weighted_area": pd.Series(
                    dtype="float64"
                ),
            })

            save_tile_checkpoint(
                empty_result,
                tile_path,
            )

            del f_tile, c_tile, empty_result
            gc.collect()
            continue

        c_tile["crop_type"] = (
            c_tile["lucode"]
            .astype("string")
            .str.strip()
            .str.upper()
            .map(CROME_LUCODE)
        )

        c_tile["crome_prob"] = (
            pd.to_numeric(
                c_tile["prob"],
                errors="coerce",
            )
            / CROME_PROBABILITY_DIVISOR
        )

        c_tile = c_tile.loc[
            c_tile["crop_type"].notna()
            & c_tile["crome_prob"].notna()
            & np.isfinite(c_tile["crome_prob"]),
            [
                "crop_type",
                "crome_prob",
                "geometry",
            ],
        ].copy()

        intersections = gpd.overlay(
            f_tile,
            c_tile,
            how="intersection",
            keep_geom_type=False,
            make_valid=True,
        )

        if intersections.empty:
            tile_result = pd.DataFrame({
                "field_id": pd.Series(
                    dtype=flame["field_id"].dtype
                ),
                "crop_type": pd.Series(
                    dtype="string"
                ),
                "_area": pd.Series(dtype="float64"),
                "_weighted_area": pd.Series(
                    dtype="float64"
                ),
            })

        else:
            intersections["_area"] = (
                intersections.geometry.area
            )

            intersections = intersections.loc[
                np.isfinite(
                    intersections["_area"]
                )
                & (
                    intersections["_area"] > 0
                )
            ].copy()

            intersections["_weighted_area"] = (
                intersections["_area"]
                * intersections["crome_prob"]
            )

            # Keep every crop class total.
            tile_result = (
                intersections
                .groupby(
                    ["field_id", "crop_type"],
                    as_index=False,
                    sort=False,
                )
                .agg(
                    _area=("_area", "sum"),
                    _weighted_area=(
                        "_weighted_area",
                        "sum",
                    ),
                )
            )

        save_tile_checkpoint(
            tile_result,
            tile_path,
        )

        print(
            f"    saved {len(tile_result):,} "
            f"field/crop totals in "
            f"{time.time() - started:.0f}s",
            flush=True,
        )

        del (
            f_tile,
            c_tile,
            intersections,
            tile_result,
        )

        gc.collect()


preflight_table = pd.DataFrame(
    preflight_records
)

if not preflight_table.empty:
    preflight_table.to_csv(
        OUT
        / (
            f"crome_preflight_"
            f"{CROME_RUN_VERSION}_"
            f"{TILE_SIZE}.csv"
        ),
        index=False,
    )

    print("\nLargest proposed CROME reads:")

    display(
        preflight_table
        .sort_values(
            "expected_crome_cells",
            ascending=False,
        )
        .head(20)
    )

if PREFLIGHT_ONLY:
    print(
        "\nPREFLIGHT ONLY completed. "
        "No CROME geometries were loaded and "
        "no intersections were performed."
    )

In [ ]:
# Combine all completed tile results
if BUILD_CROP_RESULTS and not PREFLIGHT_ONLY:

    tile_files = sorted(
        CROP_TILE_DIR.glob(
            "crop_*.parquet"
        )
    )

    if not tile_files:
        raise RuntimeError(
            "No CROME tile checkpoints were produced."
        )

    print(
        f"Combining {len(tile_files):,} "
        "tile checkpoints..."
    )

    crop_totals = pd.concat(
        [
            pd.read_parquet(path)
            for path in tile_files
        ],
        ignore_index=True,
    )

    if crop_totals.empty:
        raise RuntimeError(
            "All CROME tile results are empty."
        )

    # Combine the same field and crop across tiles.
    crop_totals = (
        crop_totals
        .groupby(
            ["field_id", "crop_type"],
            as_index=False,
            sort=False,
        )
        .agg(
            _area=("_area", "sum"),
            _weighted_area=(
                "_weighted_area",
                "sum",
            ),
        )
    )

    # Denominator for purity: total mapped CROME overlap.
    crop_totals["_total_intersected_area"] = (
        crop_totals
        .groupby("field_id")["_area"]
        .transform("sum")
    )

    crop_totals["_purity"] = (
        crop_totals["_area"]
        / crop_totals["_total_intersected_area"]
    )

    # Highest confidence-weighted area wins.
    winners = (
        crop_totals
        .sort_values(
            [
                "field_id",
                "_weighted_area",
                "_area",
                "crop_type",
            ]
        )
        .drop_duplicates(
            "field_id",
            keep="last",
        )
        .copy()
    )

    crop_result = pd.DataFrame({
        "field_id": winners["field_id"],
        "crop_type": winners["crop_type"],
        "crop_type_purity": winners["_purity"],
        "crop_prob": (
            winners["_weighted_area"]
            / winners["_area"]
        ),
    })

    if crop_result["field_id"].duplicated().any():
        raise RuntimeError(
            "Combined CROME result contains "
            "duplicate field IDs."
        )

    if "crop_2020" not in flame.columns:
        if "crop_old" in flame.columns:
            flame["crop_2020"] = flame["crop_old"]
        else:
            flame["crop_2020"] = pd.NA

    flame = (
        flame
        .drop(
            columns=[
                "crop_type",
                "crop_type_purity",
                "crop_prob",
                "crop_source",
                "lu_flame",
                "lu_crome",
                "land_use",
                "_tile_x",
                "_tile_y",
            ],
            errors="ignore",
        )
        .merge(
            crop_result,
            on="field_id",
            how="left",
        )
    )

    flame["crop_source"] = "CROME 2024"

    flame["lu_flame"] = (
        flame["crop_2020"]
        .map(crop_to_land_use)
    )

    flame["lu_crome"] = (
        flame["crop_type"]
        .map(crop_to_land_use)
    )

    flame["land_use"] = flame["lu_crome"]

    flame.to_parquet(
        PATHS["ck_crop"]
    )

    print(
        "Saved final crop checkpoint:",
        PATHS["ck_crop"],
    )

    del crop_totals, winners, crop_result
    gc.collect()

In [ ]:
# Save quality check tables and updated checkpoint

qa_summary.to_csv(
    OUT / "crome_2024_qa_summary.csv",
    index=False,
)

if (~has_crop).any():
    flame.loc[
        ~has_crop,
        [
            "field_id",
            "crop_2020",
            "geometry",
        ],
    ].to_parquet(
        OUT / "crome_2024_unmatched_fields.parquet"
    )

# Resave in case crop_2020 or analytical groups were recovered
# from an older checkpoint.
flame.to_parquet(
    PATHS["ck_crop"]
)

print(
    "\nFinal CROME dataset saved:",
    PATHS["ck_crop"],
)

In [ ]:
# Check final CROME result

EXPECTED_FLAME_ROWS = 682_623

checks = {
    "Correct row count":
        len(flame) == EXPECTED_FLAME_ROWS,

    "Unique field IDs":
        flame["field_id"].is_unique,

    "No missing field IDs":
        flame["field_id"].notna().all(),

    "Correct CRS":
        flame.crs is not None
        and flame.crs.to_epsg() == 27700,

    "Purity within 0–1":
        flame["crop_type_purity"]
        .dropna()
        .between(0, 1)
        .all(),

    "Probability within 0–1":
        flame["crop_prob"]
        .dropna()
        .between(0, 1)
        .all(),

    "land_use equals lu_crome":
        flame["land_use"]
        .fillna("__missing__")
        .eq(
            flame["lu_crome"]
            .fillna("__missing__")
        )
        .all(),

    "CROME source correct":
        flame["crop_source"]
        .dropna()
        .eq("CROME 2024")
        .all(),
}

for name, passed in checks.items():
    print(
        f"{'PASS' if passed else 'FAIL'}: {name}"
    )

matched = flame["crop_type"].notna()
complete = (
    flame["crop_type"].notna()
    & flame["crop_type_purity"].notna()
    & flame["crop_prob"].notna()
)

print("\nResult summary")
print("------------------------------")
print(f"Rows:                 {len(flame):,}")
print(f"Unique fields:        {flame['field_id'].nunique():,}")
print(f"CROME matched:        {matched.sum():,}")
print(f"CROME match rate:     {matched.mean():.2%}")
print(f"Complete results:     {complete.mean():.2%}")
print(f"Missing crop:         {(~matched).sum():,}")
print(
    "Median purity:       ",
    round(flame["crop_type_purity"].median(), 3),
)
print(
    "Median probability:  ",
    round(flame["crop_prob"].median(), 3),
)

print("\nCROME crop distribution:")
print(
    flame["crop_type"]
    .value_counts(dropna=False)
    .head(20)
    .to_string()
)

In [ ]:
region_check = (
    flame
    .assign(crome_matched=flame["crop_type"].notna())
    .groupby("region", dropna=False)
    .agg(
        fields=("field_id", "size"),
        matched=("crome_matched", "sum"),
        match_rate=("crome_matched", "mean"),
    )
    .sort_values("match_rate")
)

region_check["match_rate"] = (
    region_check["match_rate"] * 100
).round(2)

display(region_check)

In [ ]:
# Updated layer with new code assignments

# Group existing crop NAMES—no original codes required.
crop_groups = {
    "Cereal Crops": [
        "Spring Barley", "Beet", "Borage", "Buckwheat",
        "Canary Seed", "Carrot", "Celery", "Chicory",
        "Daffodil", "Dill", "Hemp", "Lettuce",
        "Spring Linseed", "Maize", "Millet", "Spring Oats",
        "Onions", "Parsley", "Parsnips", "Spring Rye",
        "Spinach", "Strawberry", "Spring Triticale",
        "Spring Wheat", "Spring Cabbage", "Turnip",
        "Spring Oilseed", "Brown Mustard", "Mustard",
        "Radish", "Potato", "Tomato", "Squash",
        "Siam Pumpkin", "Cheese Pumpkin",
        "Mixed Crop-Group 1", "Mixed Crop-Group 2",
        "Mixed Crop-Group 3", "Mixed Crop-Group 4",
        "Mixed Crop-Group 5",
        "Winter Barley", "Winter Linseed", "Winter Oats",
        "Winter Wheat", "Winter Oilseed", "Winter Rye",
        "Winter Triticale", "Winter Cabbage",
        "Coriander", "Corn Gromwell", "Phacelia",
        "German chamomile-type arable crops",
        "Corn Chamomile", "Poppy", "Hedge Bedstraw",
        "Sunflower", "Gladioli", "Sorghum", "Sweet William",
        "Italian Ryegrass", "Cover Crop"
    ],

    "Leguminous Crops": [
        "Chickpea", "Fenugreek", "Spring Field Beans",
        "Green Beans", "Lupins", "Spring Peas",
        "Soya", "Cowpea", "Lucerne", "Sainfoin", "Clover",
        "Mixed Crops-Group 1 Leguminous",
        "Mixed Crops-Group 2 Leguminous",
        "Mixed Crops-Group 3 Leguminous",
        "Winter Field Beans", "Winter Peas"
    ],

    "Grassland": [
        "Fallow Land", "Grass", "Temporary Grassland"
    ],

    "Non-Agricultural Land": [
        "Non-vegetated or Sparsely-Vegetated Land",
        "Non agriculture land",
        "Non-Agricultural Land",
        "Heather",
        "Heathland and Bracken"
    ],

    "Water": [
        "Water"
    ],

    "Trees": [
        "Perennial Crops and Isolated Trees",
        "Nursery Crops",
        "Trees and Scrubs, Short Woody Plants, Hedgerows",
        "Short Rotation Coppice"
    ]
}

def normalise_crop(series):
    return (
        series.astype("string")
        .str.normalize("NFKC")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .str.casefold()
        .replace("", pd.NA)
    )

# Ignore capitalisation differences such as "Field beans"/"Field Beans".
crop_to_land_use = {
    name.casefold(): group
    for group, names in crop_groups.items()
    for name in names
}

# Correct only the known surviving codes.
label_fixes = {
    "wa00": "Water",
    "heat": "Heather"
}

required = ["crop_2020", "crop_type"]
missing = [col for col in required if col not in flame.columns]
if missing:
    raise KeyError(f"Missing required columns: {missing}")

for col in ["crop_old", "crop_2020", "crop_type"]:
    if col not in flame.columns:
        continue

    values = flame[col].astype("string")
    corrected = normalise_crop(values).map(label_fixes)
    flame[col] = corrected.fillna(values)

# Replace the previous groups entirely.
flame["lu_flame"] = (
    normalise_crop(flame["crop_2020"])
    .map(crop_to_land_use)
)

flame["lu_crome"] = (
    normalise_crop(flame["crop_type"])
    .map(crop_to_land_use)
)

# Final analytical variable uses the 2024 grouping.
flame["land_use"] = flame["lu_crome"].copy()

# Also group crop_old when it is retained separately.
if "crop_old" in flame.columns:
    flame["crop_old_group"] = (
        normalise_crop(flame["crop_old"])
        .map(crop_to_land_use)
    )

# Check coverage and identify any names needing attention.
for crop_col, group_col in [
    ("crop_2020", "lu_flame"),
    ("crop_type", "lu_crome")
]:
    print(f"\n{crop_col} → {group_col}")
    print(flame[group_col].value_counts(dropna=False).to_string())

    unmapped = (
        normalise_crop(flame[crop_col]).notna()
        & flame[group_col].isna()
    )

    print("\nNon-missing crop names without a group:")
    print(
        flame.loc[unmapped, crop_col]
        .value_counts()
        .to_string()
    )

In [ ]:
# Updated layer with new code assignments

output_path = Path("agriculture_regrouped.gpkg")

# Protect any existing file with this name.
if output_path.exists():
    raise FileExistsError(f"File already exists: {output_path.resolve()}")

agriculture.to_file(
    output_path,
    layer="agriculture",
    driver="GPKG",
    index=False
)

print(f"Saved {len(agriculture):,} records to: {output_path.resolve()}")

### ALC land quality


In [ ]:
# ALC Labels
ALC_LABELS = {1:"Grade 1", 2:"Grade 2", 3:"Grade 3a", 4:"Grade 3b",
              5:"Grade 4", 6:"Grade 5", 7:"Non-agricultural", 8:"Urban"}

# ALC grade codes to DROP.
ALC_DROP_CODES  = {7, 8}
ALC_DROP_LABELS = {"Urban", "Non-agricultural"}

In [ ]:
# Check the grades and dataset
if PATHS["ck_alc"].exists() and not FORCE_REBUILD:
    print("ALC checkpoint already exists: skipping the raw-ALC peek.")
else:
    _alc_peek = load_vector(PATHS["alc"])
    print("ALC columns:", list(_alc_peek.columns))
    _grade_col = first_present(_alc_peek.columns, ["alcgrade","ALCGRADE","alc_grade","GRADE","Grade"])
    print("Using grade column:", _grade_col)
    print(_alc_peek[_grade_col].value_counts(dropna=False).head(12))
    del _alc_peek

In [ ]:
# Prepare the ALC layer if a regional join is required.
if PATHS["ck_alc"].exists() and not FORCE_REBUILD:
    alc = None
    print("ALC checkpoint already exists -> skipping the raw-ALC build.")
else:
    alc = load_vector(PATHS["alc"])
    _grade_col = first_present(alc.columns, ["alcgrade","ALCGRADE","alc_grade","GRADE","Grade"])
    alc = alc.rename(columns={_grade_col: "alc_code"})[["alc_code","geometry"]]
    alc = repair(alc)

    # Convert numeric grades to labels and retain existing text labels.
    alc["alc_label"] = alc["alc_code"]
    if pd.api.types.is_numeric_dtype(alc["alc_code"]):
        alc["alc_label"] = alc["alc_code"].map(ALC_LABELS).fillna(alc["alc_code"].astype(str))

    # Remove non-agricultural and urban polygons.
    before_n   = len(alc)
    drop_mask  = alc["alc_code"].isin(ALC_DROP_CODES) | alc["alc_label"].isin(ALC_DROP_LABELS)
    removed_km2 = alc.loc[drop_mask].geometry.area.sum() / 1e6
    alc = alc.loc[~drop_mask].copy()
    print(f"ALC urban/non-agricultural removed: {before_n-len(alc):,} polygons, {removed_km2:,.1f} km\u00b2")

    # Summary
    alc_removed_tab = pd.DataFrame({
        "status":["removed (urban/non-ag)","kept (agricultural)"],
        "polygons":[before_n-len(alc), len(alc)],
        "area_km2":[round(removed_km2,1),
                    round(alc.geometry.area.sum()/1e6,1)]})
    save_table(alc_removed_tab, "T3_alc_urban_removed")
    print(alc_removed_tab.to_string(index=False))

    # Use the label as the class to join on
    alc = alc.rename(columns={"alc_label":"alc_grade"})[["alc_grade","geometry"]]

In [ ]:
# Restore saved parcel classifications, or calculate them region by region.
if PATHS["ck_alc"].exists() and not FORCE_REBUILD:
    flame = gpd.read_parquet(PATHS["ck_alc"]); print("Loaded ALC checkpoint.")
else:
    alc_parts = []
    for i, reg in enumerate(REGION_LIST, 1):
        t0 = time.time()
        f_r = flame[flame["region"] == reg]
        if f_r.empty: continue
        minx, miny, maxx, maxy = f_r.total_bounds
        a_r = alc.cx[minx:maxx, miny:maxy]
        part = dominant_by_area(f_r, a_r, "alc_grade")
        alc_parts.append(part)
        print(f"[{i}] {reg:<26} {len(part):>7,} matched  {time.time()-t0:5.0f}s")
    alc_result = pd.concat(alc_parts, ignore_index=True)
    flame = (flame.drop(columns=["alc_grade","alc_grade_purity"], errors="ignore")
                  .merge(alc_result, on="field_id", how="left"))
    flame.to_parquet(PATHS["ck_alc"])
print("\nMedian ALC purity:", round(flame["alc_grade_purity"].median(), 3))
print("No parcel labelled Urban?:", not (flame["alc_grade"] == "Urban").any())
print(flame["alc_grade"].value_counts(dropna=False))

if "alc" in dir():
    del alc
    gc.collect()

In [ ]:
# Load the saved parcel classifications
if not PATHS["ck_alc"].exists():
    raise FileNotFoundError(
        "ALC checkpoint not found. Run the two original ALC cells first."
    )

flame = gpd.read_parquet(PATHS["ck_alc"])
print("Loaded ALC checkpoint.")

print("\nMedian ALC purity:", round(flame["alc_grade_purity"].median(), 3))
print("No parcel labelled Urban?:", not (flame["alc_grade"] == "Urban").any())
print(flame["alc_grade"].value_counts(dropna=False))

### Agricultural Census livestock context


In [ ]:
# Uses the existing imports and variables: Path, re, np, pd, gpd,
# PATHS, OUT, flame, ANALYSIS_CRS, and FORCE_REBUILD.

AGC_ANIMAL_CHECKPOINT = OUT / "agcensus_animal_type_v1.parquet"
FIELD_ANIMAL_CHECKPOINT = OUT / "mp_flame_animal_type_v1.parquet"
SENTINEL = -9999

DAIRY_COLUMNS = [
    "female_dairy_under_1_year",
    "female_dairy_1_2_years",
    "female_dairy_over_2_years_with_no_offspring",
    "dairy_breeding_herd_female_dairy_cattle_over_2_years_old_with_offspring",
]

BEEF_COLUMNS = [
    "female_beef_under_1_year",
    "female_beef_1_2_years",
    "female_beef_over_2_years_with_no_offspring",
    "beef_breeding_herd_female_beef_cattle_over_2_years_old_with_offspring",
    "male_cattle_under_1_year",
    "male_cattle_1_2_years",
    "male_cattle_over_2_years",
]

SHEEP_COLUMN = "total_sheep_and_lambs"
ANIMAL_COLUMNS = DAIRY_COLUMNS + BEEF_COLUMNS + [SHEEP_COLUMN]


def short_agcensus_name(name):
    return re.sub(
        r"^en-\d{4}-c[0-9a-z]+-5km-", "", name
    ).replace("-", "_")


# Read animal counts for each 5 km AgCensus cell.
if AGC_ANIMAL_CHECKPOINT.exists() and not FORCE_REBUILD:
    agc = gpd.read_parquet(AGC_ANIMAL_CHECKPOINT)
else:
    values = {}
    grid = None

    for path in sorted(Path(PATHS["agcensus_dir"]).rglob("*.shp")):
        variable = short_agcensus_name(path.stem)
        if variable not in ANIMAL_COLUMNS:
            continue

        layer = gpd.read_file(path, engine="pyogrio", use_arrow=True)

        if grid is None:
            grid = layer[["id", "geometry"]].drop_duplicates("id")

        values[variable] = (
            layer.drop_duplicates("id")
            .set_index("id")["agcval"]
            .pipe(pd.to_numeric, errors="coerce")
            .replace(SENTINEL, np.nan)
        )

    missing = sorted(set(ANIMAL_COLUMNS) - set(values))
    if missing:
        raise KeyError(f"Missing AgCensus animal variables: {missing}")
    if grid is None:
        raise FileNotFoundError("No AgCensus animal shapefiles found.")

    agc = grid.set_index("id")
    for variable, series in values.items():
        agc[variable] = series.reindex(agc.index)

    agc = gpd.GeoDataFrame(
        agc.reset_index(), geometry="geometry", crs=grid.crs
    ).to_crs(ANALYSIS_CRS)

    # Only classify a cell when all required counts are available.
    complete = agc[ANIMAL_COLUMNS].notna().all(axis=1)

    counts = pd.DataFrame({
        "Dairy cattle": agc[DAIRY_COLUMNS].sum(axis=1),
        "Beef cattle": agc[BEEF_COLUMNS].sum(axis=1),
        "Sheep": agc[SHEEP_COLUMN],
    }, index=agc.index)

    agc["animal_type"] = "Unknown"
    agc.loc[complete & counts.sum(axis=1).eq(0), "animal_type"] = (
        "No livestock"
    )

    has_animals = complete & counts.sum(axis=1).gt(0)
    agc.loc[has_animals, "animal_type"] = counts.loc[
        has_animals
    ].idxmax(axis=1)

    agc = agc[["id", "animal_type", "geometry"]]
    agc.to_parquet(AGC_ANIMAL_CHECKPOINT)


# Assign each field the animal type of its 5 km cell.
if FIELD_ANIMAL_CHECKPOINT.exists() and not FORCE_REBUILD:
    flame = gpd.read_parquet(FIELD_ANIMAL_CHECKPOINT)
else:
    flame = flame.drop(columns=["animal_type"], errors="ignore")

    field_points = gpd.GeoDataFrame(
        flame[["field_id"]].copy(),
        geometry=flame.geometry.representative_point(),
        crs=flame.crs,
    )

    field_context = gpd.sjoin(
        field_points,
        agc[["animal_type", "geometry"]].to_crs(flame.crs),
        how="left",
        predicate="within",
    )

    field_types = (
        field_context.groupby("field_id")["animal_type"]
        .first()
        .rename("animal_type")
    )
    flame = flame.join(field_types, on="field_id")

    grass_land_uses = {
        "Permanent grass",
        "Temporary grass and forage",
    }
    is_grassland = flame["land_use"].isin(grass_land_uses)

    flame.loc[is_grassland, "animal_type"] = (
        flame.loc[is_grassland, "animal_type"].fillna("Unknown")
    )
    flame.loc[~is_grassland, "animal_type"] = "Not grassland"

    flame.to_parquet(FIELD_ANIMAL_CHECKPOINT)

print(flame["animal_type"].value_counts(dropna=False))

### Physical availability and exclusions


In [ ]:
# Buffer widths (metres) turned into corridors for line features
BUFFERS = {"road": 5, "rail": 10, "watercourse": 2}

In [ ]:
# Build the exclusion layers
if (
    PATHS["exclusions"].exists() or PATHS["exclusions_legacy"].exists()
) and not FORCE_REBUILD:
    exclusion_parts = None
    _cached = (
        PATHS["exclusions"]
        if PATHS["exclusions"].exists()
        else PATHS["exclusions_legacy"]
    )
    print(f"Cached exclusions found ({_cached.name}) - skipping unchanged layers.")
    print("Set FORCE_REBUILD=True to rebuild the exclusion layers from raw data.")
else:
    exclusion_parts = []  # each standardised layer is appended here

    # --- Built-up land (polygons, WKT in a CSV, already EPSG:27700 --------------
    _bu = pd.read_csv(PATHS["built_up"])
    _bu["geometry"] = _bu["geometry"].apply(wkt.loads)
    built_up = gpd.GeoDataFrame(_bu, geometry="geometry", crs="EPSG:27700").to_crs(
        ANALYSIS_CRS
    )
    built_up = to_exclusion(built_up, "Built-up")
    exclusion_parts.append(built_up)
    quick_look(built_up, "built-up")

    # --- Protected areas (FLAGGED as their own class, not hard-removed) ----------
    protected_files = {
        "AONB": "Areas_of_Outstanding_Natural_Beauty_(England)___Natural_England.shp",
        "SSSI": "Sites_of_Special_Scientific_Interest_(England)___Natural_England.shp",
        "SAC": "Special_Areas_of_Conservation_(England)___Natural_England.shp",
        "SPA": "Special_Protection_Areas_(England)___Natural_England.shp",
        "Ramsar": "Ramsar_(England)___Natural_England.shp",
        "NNR": "National_Nature_Reserves_(England)___Natural_England.shp",
        "LNR": "Local_Nature_Reserves_(England)___Natural_England.shp",
        "NP": "National_Parks_(England)___Natural_England.shp",
    }
    _pl = []
    for desig, fn in protected_files.items():
        fp = PATHS["protected_dir"] / fn
        if not fp.exists():
            print("  (protected layer missing, skipped):", fn)
            continue
        lyr = load_vector(fp)
        lyr["designation"] = desig
        _pl.append(lyr[["designation", "geometry"]])
    if _pl:
        protected = gpd.GeoDataFrame(
            pd.concat(_pl, ignore_index=True), crs=ANALYSIS_CRS
        )
        protected = to_exclusion(protected, "Protected area (flag)")
        exclusion_parts.append(protected)
        quick_look(protected, "protected areas")

    # --- Deep peat >=30 cm (FLAG ONLY) - polygonised from the depth raster ---------
    try:
        with rasterio.open(PATHS["peat_tif"]) as src:
            clipped, transform = mask(
                src, england.to_crs(src.crs).geometry, crop=True, filled=False
            )
            depth = clipped[0]
        valid = ~np.ma.getmaskarray(depth)
        deep = valid & np.isfinite(depth.filled(np.nan)) & (depth.filled(0) >= 30.0)
        peat_geoms = [
            shape(g)
            for g, v in shapes(deep.astype("uint8"), mask=deep, transform=transform)
            if v == 1
        ]
        peat = gpd.GeoDataFrame(geometry=peat_geoms, crs=src.crs).to_crs(ANALYSIS_CRS)
        peat = to_exclusion(peat, "Deep peat >30cm (flag)")
        exclusion_parts.append(peat)
        quick_look(peat, "deep peat")
    except Exception as e:
        print("Deep-peat step skipped:", type(e).__name__, e)

    # --- Transport: roads + railways, flat buffer per feature type ---------------
    _rf = []
    for shp in Path(PATHS["road_dir"]).rglob("*.shp"):
        if "LineString" not in str(pyogrio.read_info(shp)["geometry_type"]):
            continue
        g = load_vector(shp)[["geometry"]]
        if not g.empty:
            _rf.append(g)
    if _rf:
        roads_lines = gpd.GeoDataFrame(
            pd.concat(_rf, ignore_index=True), crs=ANALYSIS_CRS
        )  # no repair() here -- these are lines, not polygons
        roads = repair(
            to_exclusion(roads_lines, "Transport", buffer_m=BUFFERS["road"])
        )  # repair() runs AFTER buffering turns them into polygons
    else:
        roads_lines = None
        roads = None
    rail = (
        to_exclusion(load_vector(PATHS["rail"]), "Transport", buffer_m=BUFFERS["rail"])
        if PATHS["rail"].exists()
        else None
    )
    transport = gpd.GeoDataFrame(
        pd.concat([g for g in [roads, rail] if g is not None], ignore_index=True),
        crs=ANALYSIS_CRS,
    )
    exclusion_parts.append(transport)
    quick_look(transport, "transport")

    # --- Open water: lakes (polygons) + watercourses (lines, buffered) -----------
    lakes = (
        to_exclusion(load_vector(PATHS["lakes"]), "Water")
        if PATHS["lakes"].exists()
        else None
    )
    wcs = (
        to_exclusion(
            load_vector(PATHS["watercourses"]), "Water", buffer_m=BUFFERS["watercourse"]
        )
        if PATHS["watercourses"].exists()
        else None
    )
    water = gpd.GeoDataFrame(
        pd.concat([g for g in [lakes, wcs] if g is not None], ignore_index=True),
        crs=ANALYSIS_CRS,
    )
    exclusion_parts.append(water)
    quick_look(water, "water")

    print(f"\nCollected {len(exclusion_parts)} exclusion layers.")

In [ ]:
# List saved exclusion files
for key in ["exclusions", "exclusions_legacy"]:
    value = PATHS.get(key)
    if value is not None:
        path = Path(value)
        if path.is_file():
            print(f"{key}: {path}\n" f"Size: {path.stat().st_size / 1e9:.2f} GB\n")

folder = Path(OUT) / "exclusion_components"

print("Saved components:")
if folder.exists():
    for path in sorted(folder.glob("*.parquet")):
        print(f"{path.name}: {path.stat().st_size / 1e9:.2f} GB")
else:
    print("No component folder found.")

In [ ]:
# Check saved component files
folder = Path(OUT) / "exclusion_components"

for name in [
    "saved_layer_01.parquet",
    "saved_layer_02.parquet",
    "saved_layer_03.parquet",
]:
    path = folder / name
    if not path.exists():
        continue

    with pq.ParquetFile(path) as parquet:
        print(f"\n{name}: {parquet.metadata.num_rows:,} records")

        if "type" in parquet.schema_arrow.names:
            batch = next(
                parquet.iter_batches(batch_size=5, columns=["type"]),
                None,
            )
            if batch is not None:
                print("Type:", batch.column(0).to_pylist())
        else:
            print("No type column")

In [ ]:
# Redo-buffer the watercourse network
COMPONENT_DIR = Path(OUT) / "exclusion_components"
COMPONENT_DIR.mkdir(parents=True, exist_ok=True)

width = float(BUFFERS["watercourse"])
if width <= 0:
    raise ValueError("Watercourse buffer must be greater than zero.")

analysis_crs = CRS.from_user_input(ANALYSIS_CRS)
if analysis_crs.to_epsg() != 27700:
    raise ValueError("This cell expects ANALYSIS_CRS = 'EPSG:27700'.")

watercourses_path = COMPONENT_DIR / f"watercourses_buffered_{width:g}m.parquet"

print(f"Buffer: {width:g} metres EACH SIDE", flush=True)


def build_watercourses():
    if watercourses_path.exists():
        print("Already saved:", watercourses_path)
        return

    source = Path(PATHS["watercourses"])
    if not source.exists():
        raise FileNotFoundError(source)

    temporary = watercourses_path.with_name(f"watercourses_{uuid.uuid4().hex}.parquet")

    geo_metadata = {
        "version": "1.0.0",
        "primary_column": "geometry",
        "columns": {
            "geometry": {
                "encoding": "WKB",
                "geometry_types": ["Polygon", "MultiPolygon"],
                "crs": analysis_crs.to_json_dict(),
            }
        },
    }

    schema = pa.schema(
        [
            ("type", pa.string()),
            ("geometry", pa.binary()),
        ],
        metadata={"geo": json.dumps(geo_metadata).encode()},
    )

    total = 0

    try:
        with pyogrio.open_arrow(
            source,
            columns=[],
            batch_size=1_000,
            use_pyarrow=True,
        ) as (metadata, reader):

            if not metadata["crs"]:
                raise ValueError("Watercourse source has no CRS.")

            geometry_column = metadata["geometry_name"] or "wkb_geometry"

            with pq.ParquetWriter(
                temporary,
                schema,
                compression="snappy",
            ) as writer:

                for number, batch in enumerate(reader, 1):
                    column = batch.schema.get_field_index(geometry_column)

                    geometry = shapely.from_wkb(
                        batch.column(column).to_numpy(zero_copy_only=False)
                    )

                    lines = gpd.GeoSeries(
                        geometry,
                        crs=metadata["crs"],
                    ).to_crs(analysis_crs)

                    lines = lines[lines.notna() & ~lines.is_empty]

                    if lines.empty:
                        continue

                    if not lines.geom_type.isin(
                        ["LineString", "MultiLineString"]
                    ).all():
                        raise ValueError("Unexpected non-line watercourse geometry.")

                    # GeoPandas defaults match the normal geometry.buffer().
                    buffered = lines.buffer(width)
                    buffered = buffered[buffered.notna() & ~buffered.is_empty]

                    if not buffered.is_valid.all():
                        raise ValueError(
                            f"Invalid buffered geometry in batch {number}."
                        )

                    table = pa.Table.from_arrays(
                        [
                            pa.array(
                                ["Water"] * len(buffered),
                                type=pa.string(),
                            ),
                            pa.array(
                                shapely.to_wkb(buffered.to_numpy()),
                                type=pa.binary(),
                            ),
                        ],
                        schema=schema,
                    )

                    writer.write_table(table)
                    total += len(buffered)

                    if number == 1 or number % 20 == 0:
                        print(
                            f"Saved {total:,} buffered watercourses...",
                            flush=True,
                        )

                    del geometry, lines, buffered, table

        if total == 0:
            raise ValueError("No watercourse features were written.")

        os.replace(temporary, watercourses_path)
        print(f"Finished: {total:,} features")
        print("Saved:", watercourses_path)

    finally:
        temporary.unlink(missing_ok=True)
        gc.collect()


build_watercourses()

In [ ]:
# Check all seven components are present
COMPONENT_DIR = Path(OUT) / "exclusion_components"

watercourse_width = float(BUFFERS["watercourse"])

watercourses_path = (
    COMPONENT_DIR / f"watercourses_buffered_{watercourse_width:g}m.parquet"
)

component_files = [
    COMPONENT_DIR / "built_up.parquet",
    COMPONENT_DIR / "protected_areas.parquet",
    COMPONENT_DIR / "saved_layer_03.parquet",  # Confirmed peat
    COMPONENT_DIR / "roads_gpkg_buffer_5m.parquet",
    COMPONENT_DIR / "rail_buffered.parquet",
    COMPONENT_DIR / "lakes.parquet",
    watercourses_path,
]

missing = [str(path) for path in component_files if not path.exists()]

if missing:
    raise FileNotFoundError("These files are not saved yet:\n" + "\n".join(missing))

combined_path = Path(OUT) / "exclusions_england_streamed.parquet"
manifest_path = combined_path.with_suffix(".manifest.json")

BATCH_SIZE = 1_000
ENGLAND_SIMPLIFY_M = 500

print("All seven saved components found.")
print("Ready to combine in small batches.")
print("Output:", combined_path)

In [ ]:
# Load the England boundary and check the cache
england_projected = england.to_crs(ANALYSIS_CRS)

england_geometry = shapely.union_all(england_projected.geometry.to_numpy())

if ENGLAND_SIMPLIFY_M > 0:
    england_geometry = shapely.simplify(
        england_geometry,
        ENGLAND_SIMPLIFY_M,
        preserve_topology=True,
    )

shapely.prepare(england_geometry)

del england_projected
gc.collect()


def read_component_batches(path):
    with pq.ParquetFile(path) as parquet:
        metadata = parquet.schema_arrow.metadata or {}

        if b"geo" not in metadata:
            raise ValueError(f"Missing GeoParquet metadata: {path}")

        geo = json.loads(metadata[b"geo"])
        geometry_column = geo["primary_column"]
        source_crs = geo["columns"][geometry_column].get("crs")

        if source_crs is None:
            raise ValueError(f"Missing explicit CRS: {path}")

        for batch in parquet.iter_batches(
            batch_size=BATCH_SIZE,
            columns=["type", geometry_column],
        ):
            data = batch.to_pandas()

            geometry = shapely.from_wkb(data.pop(geometry_column).to_numpy())

            chunk = gpd.GeoDataFrame(
                data,
                geometry=geometry,
                crs=source_crs,
            ).to_crs(ANALYSIS_CRS)

            del data, geometry
            yield chunk
            del chunk


# Record which inputs and settings produced the combined file.
build_spec = {
    "version": 2,
    "inputs": [
        {
            "path": str(path.resolve()),
            "size": path.stat().st_size,
            "mtime_ns": path.stat().st_mtime_ns,
        }
        for path in component_files
    ],
    "crs": str(ANALYSIS_CRS),
    "watercourse_buffer_m": watercourse_width,
    "england_simplify_m": ENGLAND_SIMPLIFY_M,
    "england_geometry": shapely.to_wkb(
        england_geometry,
        hex=True,
    ),
}

saved_manifest = None

if manifest_path.exists():
    try:
        saved_manifest = json.loads(manifest_path.read_text())
    except (ValueError, OSError):
        pass

reuse_combined = (
    combined_path.exists()
    and saved_manifest is not None
    and saved_manifest.get("build_spec") == build_spec
    and not globals().get("FORCE_REBUILD", False)
)

print("Boundary ready.")
print("Batch size:", BATCH_SIZE)
print("Reuse completed combined file:", reuse_combined)

In [ ]:
# Combine all components into the final exclusions parquet
if reuse_combined:
    feature_count = saved_manifest["feature_count"]
    area_by_type = saved_manifest["area_km2"]

    print(f"Using completed cache: {feature_count:,} features")

else:
    temporary_path = combined_path.with_name(
        f"{combined_path.stem}_{uuid.uuid4().hex}.parquet"
    )

    output_crs = gpd.GeoSeries([], crs=ANALYSIS_CRS).crs

    geo_metadata = {
        "version": "1.0.0",
        "primary_column": "geometry",
        "columns": {
            "geometry": {
                "encoding": "WKB",
                "geometry_types": ["Polygon", "MultiPolygon"],
                "crs": output_crs.to_json_dict(),
            }
        },
    }

    schema = pa.schema(
        [
            ("type", pa.string()),
            ("geometry", pa.binary()),
        ],
        metadata={"geo": json.dumps(geo_metadata).encode()},
    )

    feature_count = 0
    area_by_type = defaultdict(float)
    started = time.perf_counter()

    # Only a successfully finished ones are built.
    manifest_path.unlink(missing_ok=True)

    try:
        with pq.ParquetWriter(
            temporary_path,
            schema,
            compression="snappy",
        ) as writer:

            for path in component_files:
                component_count = 0
                component_start = time.perf_counter()

                print(f"\nProcessing {path.name}...", flush=True)

                for batch_number, chunk in enumerate(
                    read_component_batches(path),
                    start=1,
                ):
                    geometry = chunk.geometry.to_numpy()

                    valid = ~shapely.is_missing(geometry) & ~shapely.is_empty(geometry)

                    keep = np.zeros(len(chunk), dtype=bool)
                    keep[valid] = shapely.intersects(
                        england_geometry,
                        geometry[valid],
                    )

                    selected = chunk.loc[keep, ["type", "geometry"]].reset_index(
                        drop=True
                    )

                    if not selected.empty:
                        if not selected.geom_type.isin(
                            ["Polygon", "MultiPolygon"]
                        ).all():
                            raise ValueError(f"Non-polygon geometry in {path.name}")

                        areas = pd.DataFrame(
                            {
                                "type": selected["type"].to_numpy(),
                                "km2": (selected.geometry.area.to_numpy() / 1e6),
                            }
                        )

                        totals = areas.groupby("type")["km2"].sum()

                        for label, area in totals.items():
                            area_by_type[str(label)] += float(area)

                        table = pa.Table.from_arrays(
                            [
                                pa.array(
                                    selected["type"].tolist(),
                                    type=pa.string(),
                                ),
                                pa.array(
                                    shapely.to_wkb(selected.geometry.to_numpy()),
                                    type=pa.binary(),
                                ),
                            ],
                            schema=schema,
                        )

                        writer.write_table(table)

                        component_count += len(selected)
                        feature_count += len(selected)

                        del table, areas, totals

                    del selected, geometry, chunk, keep, valid

                    if batch_number % 100 == 0:
                        print(
                            f"  {batch_number:,} batches processed; "
                            f"{component_count:,} features retained",
                            flush=True,
                        )
                        gc.collect()

                print(
                    f"  Finished {path.name}: "
                    f"{component_count:,} features in "
                    f"{time.perf_counter() - component_start:.1f}s",
                    flush=True,
                )

                gc.collect()

        os.replace(temporary_path, combined_path)

        manifest = {
            "build_spec": build_spec,
            "feature_count": feature_count,
            "area_km2": dict(area_by_type),
        }

        temporary_manifest = manifest_path.with_name(manifest_path.name + ".tmp")
        temporary_manifest.write_text(json.dumps(manifest))
        os.replace(temporary_manifest, manifest_path)

    finally:
        temporary_path.unlink(missing_ok=True)

    print(
        f"\nCompleted: {feature_count:,} features in "
        f"{(time.perf_counter() - started) / 60:.1f} minutes"
    )

PATHS["exclusions"] = combined_path
print("Saved file:", combined_path)

#### Apply exclusions to each field


In [ ]:
import time
start = time.perf_counter()

In [ ]:
HARD_EXCL = ["Built-up", "Water", "Transport"]

# Release large intermediates left by a previous attempt.
for name in ("excl", "hard", "frac"):
    globals().pop(name, None)
gc.collect()

if agriculture.crs is None or not agriculture.crs.is_projected:
    raise ValueError("Use a suitable projected CRS for area calculations.")

# Filter BEFORE loading/repairing all exclusions.
hard = gpd.read_parquet(
    PATHS["exclusions"],
    columns=["geometry"],
    filters=[("type", "in", HARD_EXCL)],
)
if hard.crs is None:
    raise ValueError("Exclusions have no CRS.")
if hard.crs != agriculture.crs:
    hard = hard.to_crs(agriculture.crs)

hard = repair(hard)
hard = hard.loc[hard.geometry.notna() & ~hard.geometry.is_empty]

geoms = hard.geometry.to_numpy()
tree = shapely.STRtree(geoms)
fractions = np.zeros(len(agriculture), dtype=np.float64)

for i, field in enumerate(agriculture.geometry):
    if field is None or field.is_empty:
        continue
    if not field.is_valid:
        field = shapely.make_valid(field)

    area = field.area
    if area <= 0:
        continue

    hits = tree.query(field, predicate="intersects")
    if len(hits):
        # Clip first, then union so overlapping exclusions count only once.
        pieces = shapely.intersection(geoms[hits], field)
        fractions[i] = np.clip(
            shapely.union_all(pieces).area / area, 0.0, 1.0
        )
        del pieces
    del hits
    if i == 999:
        elapsed = time.perf_counter() - start
        estimated = elapsed * len(agriculture) / 10000
        print(
            f"First 1,000 fields: {elapsed:.1f}s | "
            f"Estimated total loop: {estimated / 60:.1f} min"
        )

    if (i + 1) % 10_000 == 0:
        print(f"Processed {i + 1:,}/{len(agriculture):,} fields", flush=True)

# Positional assignment avoids a separate field_id mapping.
agriculture["excluded_frac"] = fractions
agriculture["available_ha"] = (
    agriculture["size_ha"] * (1 - agriculture["excluded_frac"])
).clip(lower=0)
agriculture["excluded"] = agriculture["excluded_frac"] >= 0.90

del tree, geoms, hard
gc.collect()

print(f"Mean excluded fraction: {agriculture['excluded_frac'].mean()*100:.1f}%")
print(
    f"Fields ~fully excluded (>=90%): {int(agriculture['excluded'].sum()):,} "
    f"({agriculture['excluded'].mean()*100:.1f}%)"
)
print(
    f"Total available area: {agriculture['available_ha'].sum():,.0f} ha "
    f"of {agriculture['size_ha'].sum():,.0f} ha"
)

### RPA coverage check


In [ ]:
try:
    rpa = load_vector(PATHS["rpa_points"])
    ref = first_present(rpa.columns, ["parcel_ref","PARCEL_REF","SHEET_PARC","REF","id","OBJECTID"])
    rpa = rpa.rename(columns={ref:"rpa_parcel_ref"}) if ref else rpa.assign(rpa_parcel_ref=np.arange(len(rpa)))
    pip = gpd.sjoin(rpa[["rpa_parcel_ref","geometry"]], flame[["field_id","geometry"]],
                    how="left", predicate="within")
    print(f"RPA parcels captured by a FLAME field: {pip['field_id'].notna().mean()*100:.1f}% of {len(pip):,}")
    cov = (gpd.sjoin(rpa[["rpa_parcel_ref","geometry"]], regions, how="left", predicate="within")
             .assign(captured=lambda d: d.index.isin(pip.dropna(subset=["field_id"]).index))
             .groupby("region")["captured"].agg(rpa_parcels="count", captured="sum").reset_index())
    cov["capture_rate_%"] = (cov["captured"]/cov["rpa_parcels"]*100).round(1)
    save_table(cov, "rpa_coverage_by_region")
    print(cov.to_string(index=False))
except Exception as e:
    print("RPA step skipped:", type(e).__name__, e)

In [ ]:
try:
    save_table(cov, "rpa_coverage_by_region")
    print(cov.to_string(index=False))
except NameError:
    print("RPA coverage not available.")

## Final field-level dataset


In [ ]:
# Check the 'flame' dataframe before exporting
data = flame
geom_col = data.geometry.name
attrs = data.drop(columns=[geom_col])

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 100)

print(f"RECORDS: {len(data):,}")
print(f"COLUMNS: {len(data.columns)}")
print(f"CRS: {data.crs}")
print(f"GEOMETRY COLUMN: {geom_col}")
print(f"MEMORY: {data.memory_usage(deep=True).sum() / 1024**2:,.1f} MB")

# Complete data structure
structure = []
for col in data.columns:
    s = data[col]
    is_geometry = col == geom_col
    values = s.geom_type if is_geometry else s
    examples = values.dropna().astype(str).drop_duplicates().head(5)

    structure.append({
        "column": col,
        "dtype": str(s.dtype),
        "non_null": int(s.notna().sum()),
        "null": int(s.isna().sum()),
        "null_%": round(s.isna().mean() * 100, 2),
        "distinct_non_null": (
            pd.NA if is_geometry else s.nunique(dropna=True)
        ),
        "examples": " | ".join(examples),
    })

print("\nCOMPLETE COLUMN STRUCTURE")
display(pd.DataFrame(structure))

# Numeric distributions
numeric = attrs.select_dtypes(include="number")
print("\nNUMERIC STATISTICS")
if not numeric.empty:
    display(numeric.describe(
        percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]
    ).T)

# Common values: text/category columns
print("\nCOMMON VALUES — TOP 10 PER COLUMN")
for col in attrs.columns:
    s = attrs[col]
    if not pd.api.types.is_numeric_dtype(s) or s.nunique() <= 20:
        counts = s.value_counts(dropna=False).head(10)
        print(f"\n{col} ({s.nunique():,} distinct non-null values)")
        display(pd.DataFrame({
            "count": counts,
            "% of rows": (counts / len(data) * 100).round(2)
        }))

# Basic validation
checks = []

def check(label, mask):
    checks.append({
        "check": label,
        "flagged_rows": int(mask.fillna(False).sum())
    })

if "field_id" in data:
    check("Missing field_id", data["field_id"].isna())
    check("Rows sharing a non-null field_id",
          data["field_id"].notna()
          & data["field_id"].duplicated(keep=False))

for col in ["size_ha", "area_flame", "grass_area_ha",
            "total_lsu", "stocking_lsu_per_grass_ha"]:
    if col in numeric:
        check(f"{col}: negative values", data[col] < 0)
        check(f"{col}: infinite values", np.isinf(data[col]))

if "size_ha" in numeric:
    check("size_ha: zero area", data["size_ha"] == 0)

if "alc_grade_purity" in numeric:
    s = data["alc_grade_purity"]
    check("alc_grade_purity: outside 0–1",
          s.notna() & ~s.between(0, 1))

geometry_present = data.geometry.notna()
check("Missing geometry", ~geometry_present)
check("Empty geometry", geometry_present & data.geometry.is_empty)
check("Invalid geometry", geometry_present & ~data.geometry.is_valid)

required = {"total_lsu", "grass_area_ha", "stocking_lsu_per_grass_ha"}
if required.issubset(numeric.columns):
    usable = data[list(required)].notna().all(axis=1)
    usable &= data["grass_area_ha"] > 0
    expected = data["total_lsu"] / data["grass_area_ha"].where(
        data["grass_area_ha"] > 0
    )
    mismatch = ~np.isclose(
        data["stocking_lsu_per_grass_ha"], expected,
        rtol=0.001, atol=0.000001, equal_nan=True
    )
    check("Stocking rate differs from LSU / grass area",
          usable & mismatch)
    print(f"\nStocking-rate rows tested: {usable.sum():,}")

print("\nVALIDATION FLAGS — investigate - not all flags imply errors")
display(pd.DataFrame(checks))

print("\nGEOMETRY TYPES")
display(data.geometry.geom_type.value_counts(dropna=False))

# Compare recorded area with boundary area in this dataset's metre CRS
if data.crs is not None and data.crs.to_epsg() == 27700:
    if "size_ha" in numeric:
        geometry_ha = data.geometry.area / 10_000
        usable = data["size_ha"] > 0
        difference_pct = (
            (geometry_ha - data["size_ha"])
            / data["size_ha"].where(usable) * 100
        )
        print("\nGEOMETRY AREA DIFFERENCE FROM size_ha (%)")
        display(difference_pct.describe(
            percentiles=[0.01, 0.5, 0.99]
        ))
        print("Rows differing by more than 1%:",
              int((difference_pct.abs() > 1).sum()))

# Classification and farm-level consistency
if {"crop_old", "land_use"}.issubset(data.columns):
    print("\nORIGINAL CROP × CURRENT LAND USE — ROW COUNTS")
    display(pd.crosstab(
        data["crop_old"].fillna("<missing>"),
        data["land_use"].fillna("<missing>")
    ))

if "entity_id" in data:
    print(f"\nUNIQUE ENTITIES: {data['entity_id'].nunique():,}")
    for col in ["grass_area_ha", "total_lsu", "farm_type"]:
        if col in data:
            distinct = data.groupby("entity_id")[col].nunique()
            print(
                f"Entities with multiple non-null {col} values: "
                f"{(distinct > 1).sum():,}"
            )

In [ ]:
final_cols = [
    "field_id","farm_id","farm_name","farm_type","land_use",
    "crop_type","crop_type_purity","crop_prob","crop_source","crop_old",
    "alc_grade","alc_grade_purity",
    "stocking_lu_ha","grazing_intensity","livestock_system",
    "livestock_data_quality","grass_area_ha",
    "flame_place_type","flame_livestock_hint",
    "size_ha","available_ha","excluded_frac","excluded",
    "region","local_authority","catchment",
    "completeness_score","geometry",
]
agriculture = flame[[c for c in final_cols if c in flame.columns]].copy()
agriculture.to_file(OUT / "agriculture.gpkg", driver="GPKG")
print("Saved:", OUT / "agriculture.gpkg")
print("Fields:", f"{len(agriculture):,}", "| total ha:", round(agriculture['size_ha'].sum()),
      "| available ha:", round(agriculture['available_ha'].sum()))
agriculture.head()